In [ ]:
import pandas as pd

In [ ]:
jkt_temp = pd.read_csv("data/jakarta-temperature.csv", skiprows=3)
mks_temp = pd.read_csv("data/makassar-temperature.csv", skiprows=3)
sby_temp = pd.read_csv("data/surabaya-temperature.csv", skiprows=3)

In [ ]:
jkt_temp['Date'] = pd.to_datetime(jkt_temp['time'])
mks_temp['Date'] = pd.to_datetime(mks_temp['time'])
sby_temp['Date'] = pd.to_datetime(sby_temp['time'])

jkt_temp.set_index("Date", inplace=True)
mks_temp.set_index("Date", inplace=True)
sby_temp.set_index("Date", inplace=True)

In [ ]:
import matplotlib.pyplot as plt


jkt_monthly = jkt_temp.resample("YE").mean(numeric_only=True)
mks_monthly = mks_temp.resample("YE").mean(numeric_only=True)
sby_monthly = sby_temp.resample("YE").mean(numeric_only=True)

plt.figure(figsize=(10, 5.5), dpi=100)

plt.plot(jkt_monthly.index, jkt_monthly['temperature_2m (°C)'],   color='blue',   linewidth=2.2, label='jakarta')
plt.plot(mks_monthly.index, mks_monthly['temperature_2m (°C)'],   color='red',    linewidth=2.2, label='makassar')
plt.plot(sby_monthly.index, sby_monthly['temperature_2m (°C)'],  color='green',  linewidth=2.2, label='surabaya')

plt.title('(b)', fontsize=13, fontweight='bold', loc='left', pad=10)
plt.ylabel('Temperature (°C)', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.7)

# Date formatting
# plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=7))
# plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %d\n%Y'))
plt.xticks(rotation=0, ha='center')

# plt.ylim(24, 36)
# Legend
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3, fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import requests
import time
from math import radians, cos, sin, asin, sqrt
import random

lat_min = -8.9
lat_max = -5.8
lon_min = 105.0
lon_max = 114.5

grid_resolution = 0.5
min_distance_deg = 0.7
target_sample_size = 50

random_seed = 42
random.seed(random_seed)

# =========================
# HELPER FUNCTIONS
# =========================

def haversine(lat1, lon1, lat2, lon2):
    """
    Compute distance between two lat/lon points in km.
    """
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    return R * c

def is_far_enough(candidate, selected, min_deg):
    for lat2, lon2, _ in selected:
        if abs(candidate[0] - lat2) < min_deg and abs(candidate[1] - lon2) < min_deg:
            return False
    return True

def get_elevation(lat, lon):
    url = f"https://api.open-meteo.com/v1/elevation?latitude={lat}&longitude={lon}"
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        return data["elevation"][0]
    except:
        return None

# =========================
# STEP 1 — CREATE GRID
# =========================

lat_points = np.arange(lat_min, lat_max, grid_resolution)
lon_points = np.arange(lon_min, lon_max, grid_resolution)

grid = [(lat, lon) for lat in lbt_points for lon in lon_points]

print(f"Initial grid points: {len(grid)}")

# =========================
# STEP 2 — QUERY ELEVATION
# =========================

points_with_elevation = []

for lat, lon in grid:
    elevation = get_elevation(lat, lon)
    time.sleep(0.2)  # avoid API overload
    
    if elevation is None:
        continue
    
    if elevation == 0:
        continue  # remove ocean
    
    points_with_elevation.append((lat, lon, elevation))

print(f"Land points after filtering ocean: {len(points_with_elevation)}")

# =========================
# STEP 3 — ELEVATION STRATIFICATION
# =========================

coastal = []
mid = []
highland = []

for lat, lon, elev in points_with_elevation:
    if elev < 50:
        coastal.append((lat, lon, elev))
    elif elev < 500:
        mid.append((lat, lon, elev))
    else:
        highland.append((lat, lon, elev))

print("Elevation bins:")
print("Coastal:", len(coastal))
print("Mid:", len(mid))
print("Highland:", len(highland))

# =========================
# STEP 4 — BALANCED SAMPLING
# =========================

selected_points = []

def sample_from_bin(bin_points, n_needed):
    sampled = []
    random.shuffle(bin_points)
    for candidate in bin_points:
        if len(sampled) >= n_needed:
            break
        if is_far_enough(candidate, selected_points + sampled, min_distance_deg):
            sampled.append(candidate)
    return sampled

n_coastal = int(target_sample_size * 0.3)
n_mid = int(target_sample_size * 0.4)
n_high = target_sample_size - n_coastal - n_mid

selected_points += sample_from_bin(coastal, n_coastal)
selected_points += sample_from_bin(mid, n_mid)
selected_points += sample_from_bin(highland, n_high)

print(f"Selected points: {len(selected_points)}")

# =========================
# STEP 5 — SAVE RESULTS
# =========================

df = pd.DataFrame(selected_points, columns=["latitude", "longitude", "elevation"])
df.to_csv("selected_stations.csv", index=False)

print("Saved selected stations to selected_stations.csv")
print(df)

In [ ]:
import numpy as np
import pandas as pd
import requests
import time
from math import radians, cos, sin, asin, sqrt
import random

lat_min = -10.5
lat_max = 1.5
lon_min = 130.5
lon_max = 141.0

grid_resolution = 0.25
min_distance_deg = 0.7
target_sample_size = 50

random_seed = 42
random.seed(random_seed)

# =========================
# HELPER FUNCTIONS
# =========================

def haversine(lat1, lon1, lat2, lon2):
    """
    Compute distance between two lat/lon points in km.
    """
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    return R * c

def is_far_enough(candidate, selected, min_deg):
    for lat2, lon2, _ in selected:
        if abs(candidate[0] - lat2) < min_deg and abs(candidate[1] - lon2) < min_deg:
            return False
    return True

def get_elevation(lat, lon):
    url = f"https://api.open-meteo.com/v1/elevation?latitude={lat}&longitude={lon}"
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        return data["elevation"][0]
    except:
        return None

# =========================
# STEP 1 — CREATE GRID
# =========================

lat_points = np.arange(lat_min, lat_max, grid_resolution)
lon_points = np.arange(lon_min, lon_max, grid_resolution)

grid = [(lat, lon) for lat in lat_points for lon in lon_points]

print(f"Initial grid points: {len(grid)}")

# =========================
# STEP 2 — QUERY ELEVATION
# =========================

points_with_elevation = []

for lat, lon in grid:
    elevation = get_elevation(lat, lon)
    time.sleep(0.2)  # avoid API overload
    
    if elevation is None:
        continue
    
    if elevation == 0:
        continue  # remove ocean
    
    points_with_elevation.append((lat, lon, elevation))

print(f"Land points after filtering ocean: {len(points_with_elevation)}")

# =========================
# STEP 3 — ELEVATION STRATIFICATION
# =========================

coastal = []
mid = []
highland = []

for lat, lon, elev in points_with_elevation:
    if elev < 50:
        coastal.append((lat, lon, elev))
    elif elev < 500:
        mid.append((lat, lon, elev))
    else:
        highland.append((lat, lon, elev))

print("Elevation bins:")
print("Coastal:", len(coastal))
print("Mid:", len(mid))
print("Highland:", len(highland))

# =========================
# STEP 4 — BALANCED SAMPLING
# =========================

selected_points = []

def sample_from_bin(bin_points, n_needed):
    sampled = []
    random.shuffle(bin_points)
    for candidate in bin_points:
        if len(sampled) >= n_needed:
            break
        if is_far_enough(candidate, selected_points + sampled, min_distance_deg):
            sampled.append(candidate)
    return sampled

n_coastal = int(target_sample_size * 0.3)
n_mid = int(target_sample_size * 0.4)
n_high = target_sample_size - n_coastal - n_mid

selected_points += sample_from_bin(coastal, n_coastal)
selected_points += sample_from_bin(mid, n_mid)
selected_points += sample_from_bin(highland, n_high)

print(f"Selected points: {len(selected_points)}")

# =========================
# STEP 5 — SAVE RESULTS
# =========================

df = pd.DataFrame(selected_points, columns=["latitude", "longitude", "elevation"])
df.to_csv("papua_selected_stations_grid.csv", index=False)

print("Saved selected stations to selected_stations.csv")
print(df)

In [ ]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": [-8.7, -8.4, -8.3, -8.4, -8.5, -9.5, -9.7, -10, -9.5, -8.6, -9.4, -8.5],
	"longitude": [122.5, 122.6, 120.3, 120.8, 119.9, 119.6, 120.4, 120.4, 119, 121.6, 119.9, 118.7],
	"hourly": "temperature_2m",
}
responses = openmeteo.weather_api(url, params=params)



# Process 12 locations
for response in responses:
	latitude = response.Latitude()
	longtitude = response.Longitude()
	print(f"\nCoordinates: {latitude}°N {longtitude}°E")
	print(f"Elevation: {response.Elevation()} m asl")
	print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")
	
	# Process hourly data. The order of variables needs to be the same as requested.
	hourly = response.Hourly()
	hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
	
	hourly_data = {"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"),
		"latitude" : latitude,
		"longtitude" : longtitude
	}
	
	hourly_data["temperature_2m"] = hourly_temperature_2m
	
	hourly_dataframe = pd.DataFrame(data = hourly_data)
	print("\nHourly data\n", hourly_dataframe)
	

### Merged Temp & Station Dataframe

In [ ]:
max_temp_df = pd.read_csv("data/sumatra/sumatra-max-temp.csv")
station_df = pd.read_csv("data/sumatra/sumatra-station.csv")

In [ ]:
merged_df = max_temp_df.merge( # join the datasets
    station_df[['location_id', 'latitude', 'longitude', 'elevation']],
    on="location_id", # matching keys
    how='left' # keep all temperature rows
)
merged_df.to_csv("data/sumatra/sumatra-merged.csv", index=False)

### Convert Station Names

In [ ]:
def convert_station_code(df, prefix):
    df['location_id'] = prefix + (df['location_id'] + 1).astype(str).str.zfill(2)
    return df

In [ ]:
print("test")

In [ ]:
import pandas as pd
java_df = pd.read_csv("data/java/java-merged.csv")
maluku_df = pd.read_csv("data/maluku/maluku-merged.csv")
ntt_df = pd.read_csv("data/ntt/ntt-merged.csv")
papua_df = pd.read_csv("data/ntt/ntt-merged.csv")
sumatra_df = pd.read_csv("data/ntt/ntt-merged.csv")

java_df = convert_station_code(java_df, "J")
maluku_df = convert_station_code(maluku_df, "M")
ntt_df = convert_station_code(ntt_df, "N")
papua_df = convert_station_code(papua_df, "P")
sumatra_df = convert_station_code(sumatra_df, "S")

### Get Temporal Feature

In [ ]:
import numpy as np
def get_solar_declination(df):
    df['time'] = pd.to_datetime(df['time'])
    df['year'] = df['time'].dt.year
    df['doy'] = df['time'].dt.dayofyear
    df["sin_doy"] = np.sin(2 * np.pi * df["doy"] / 365) # tells us where we are in the yearly seasonal cycle
    df["cos_doy"] = np.cos(2 * np.pi * df["doy"] / 365)

    df["solar_declination"] = 23.44 * np.sin( # how strong is the position today 
        2 * np.pi * (284 + df["doy"]) / 365
    )
    return df


In [ ]:
java_df = get_solar_declination(java_df)
maluku_df = get_solar_declination(maluku_df)
ntt_df = get_solar_declination(ntt_df)
papua_df = get_solar_declination(papua_df)
sumatra_df = get_solar_declination(sumatra_df)

### ENSO & Dipole Index

In [ ]:
java_df.info()

In [ ]:
dmi_df = pd.read_csv("data/dmieast.had.long.csv")
enso_df = pd.read_csv("data/nina34.anom.csv")
# def get_enso_dmi(df, dmi_df, enso_df):
enso_df.info()

In [ ]:
dmi_df['Date'] = pd.to_datetime(dmi_df['Date'], dayfirst=True)
dmi_df['Date'].dt.month

In [ ]:
java_df

In [ ]:
def get_enso_dmi(df, dmi_df, enso_df):
    df["month"] = df['time'].dt.month
    df["year"] = df['time'].dt.year


    dmi_df['Date'] = pd.to_datetime(dmi_df['Date'], dayfirst=True)
    dmi_df['month'] = dmi_df['Date'].dt.month
    dmi_df["year"] = dmi_df['Date'].dt.year
    
    enso_df['Date'] = pd.to_datetime(enso_df['Date'], dayfirst=True)
    enso_df['month'] = enso_df["Date"].dt.month
    enso_df['year'] = enso_df["Date"].dt.year

    merge_df = df.merge(
        dmi_df[["year", "month", 'DMI EAST']],
        on=["year", "month"],
        how="left"
    )
    
    merge_df = merge_df.merge(
        enso_df[["year", "month", 'Nino Anom 3.4']],
        on=["year", "month"],
        how="left"
    )
    return merge_df

In [ ]:
java_complete_df = get_enso_dmi(java_df, dmi_df, enso_df)
maluku_complete_df = get_enso_dmi(maluku_df, dmi_df, enso_df)
ntt_complete_df = get_enso_dmi(ntt_df, dmi_df, enso_df)
papua_complete_df = get_enso_dmi(papua_df, dmi_df, enso_df)
sumatra_complete_df = get_enso_dmi(sumatra_df, dmi_df, enso_df)

In [ ]:
java_complete_df['region'] = "java"
maluku_complete_df['region'] = "maluku"
ntt_complete_df['region'] = "ntt"
papua_complete_df['region'] = "papua"
sumatra_complete_df['region'] = "sumatra"

In [ ]:
for island in [java_complete_df,maluku_complete_df, ntt_complete_df, papua_complete_df, sumatra_complete_df]:
    print("="*40)
    print(island.isna().sum())
    print("="*40)

In [ ]:
# delete 2026-01-01 data -> DMi East value not exist yet
for island in [java_complete_df,maluku_complete_df, ntt_complete_df, papua_complete_df, sumatra_complete_df]:
    island.dropna(subset=['DMI EAST'], axis=0, inplace=True)

In [ ]:
df_final = pd.concat([java_complete_df, maluku_complete_df, ntt_complete_df, papua_complete_df, sumatra_complete_df], axis=0)


In [ ]:
import numpy as np
indices = np.where(java_complete_df['time'] == "2025-04-30")[0]
indices

In [ ]:
java_complete_df.head()

In [ ]:
# remove data that did not have DMI East [2025-05-01 - 2026-01-01]
def filter_date(df):
    df_filtered = df[
        ~df['time'].between("2025-05-01", "2026-01-01")
    ]
    return df_filtered

java_complete_df = filter_date(java_complete_df)
maluku_complete_df = filter_date(maluku_complete_df)
ntt_complete_df = filter_date(ntt_complete_df)
papua_complete_df = filter_date(papua_complete_df)
sumatra_complete_df = filter_date(sumatra_complete_df)

java_complete_df.to_csv("data/java/java_complete_dataset.csv", index=False)
maluku_complete_df.to_csv("data/maluku/maluku_complete_dataset.csv", index=False)
ntt_complete_df.to_csv("data/ntt/ntt_complete_dataset.csv", index=False)
papua_complete_df.to_csv("data/papua/papua_complete_dataset.csv", index=False)
sumatra_complete_df.to_csv("data/java/java_complete_dataset.csv", index=False)

In [ ]:
df = pd.concat([java_complete_df, maluku_complete_df, ntt_complete_df, papua_complete_df, sumatra_complete_df], axis=0)
df.to_csv("data/test.csv", index=False)

In [ ]:
df.info()

In [ ]:
df.rename(columns={'temperature_2m_max (°C)':"max_temperature"}, inplace=True)
df.rename(columns=str.lower, inplace=True)

In [ ]:
df[['max_temperature', 'latitude', "longitude", "sin_doy", "cos_doy", "solar_declination", "dmi east", "nino anom 3.4"]] = df[['max_temperature', 'latitude', "longitude", "sin_doy", "cos_doy", "solar_declination", "dmi east", "nino anom 3.4"]].astype('float32')
df[['year', 'doy']] = df[['year','doy']].astype("int16")
df['month'] = df['month'].astype('int8')

In [ ]:
df.to_csv("data/merged_dataset.csv", index=False)

# Domain Adaptation

In [ ]:
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

import torch
import random
import os

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

USE_AMP = (DEVICE.type == "cuda" and os.environ.get("TSF_USE_AMP", "1") != "0")
USE_TORCH_COMPILE = (os.environ.get("TSF_TORCH_COMPILE", "0") == "1")

In [ ]:
TARGET_COL = "max_temperature"
FEATURE_COLS = [
    "max_temperature",
    "latitude",
    "longitude",
    "elevation",
    "sin_doy",
    "cos_doy",
    "solar_declination",
    "dmi east",
    "nino anom 3.4",
]
INPUT_LEN = 14
HORIZON = 7
STRIDE = 2


LOCAL_TEMPORAL_COLS = ["max_temperature", "sin_doy", "cos_doy", "solar_declination"]
GEO_COLS = ["latitude", "longitude", "elevation"]
CLIMATE_COLS = ["nino anom 3.4", "dmi east"]


### GPU Utils

In [ ]:
_CPU_CORES = os.cpu_count() or 4
# Threadripper 3960X = 24 cores (48 threads). For NumPy/torch this is a good default.
DEFAULT_CPU_THREADS = min(24, _CPU_CORES)

# Only set if user hasn't already configured these in their environment.
os.environ.setdefault("OMP_NUM_THREADS", str(DEFAULT_CPU_THREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(DEFAULT_CPU_THREADS))


import random
import torch



torch.set_default_dtype(torch.float32)
torch.set_num_threads(DEFAULT_CPU_THREADS)
torch.set_num_interop_threads(max(1, min(4, DEFAULT_CPU_THREADS // 2)))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda":
    # Ampere GPU (3090 Ti): TF32 is a big speedup for matmul/conv with minimal impact for this task.
    print(f"Training With: {DEVICE}")
    _use_tf32 = (os.environ.get("TSF_USE_TF32", "1") != "0")
    if _use_tf32:
        try:
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32 = True
        except Exception:
            pass
    torch.backends.cudnn.benchmark = True

USE_AMP = (DEVICE.type == "cuda" and os.environ.get("TSF_USE_AMP", "1") != "0")
USE_TORCH_COMPILE = (os.environ.get("TSF_TORCH_COMPILE", "0") == "1")


def _maybe_compile(model: torch.nn.Module) -> torch.nn.Module:
    if not USE_TORCH_COMPILE:
        return model
    try:
        return torch.compile(model)  # PyTorch 2.x
    except Exception:
        return model


def _dataloader_kwargs(shuffle: bool, drop_last: bool = False):
    """
    Sensible DataLoader defaults for this workstation.
    - Use CPU workers to overlap preprocessing/host->device transfers.
    - Use pin_memory for faster H2D copies when using CUDA.
    """
    num_workers = 0
    if DEVICE.type == "cuda":
        # Windows multiprocessing can be heavier; keep this conservative but non-zero.
        num_workers = min(8, max(2, DEFAULT_CPU_THREADS // 3))

    kwargs = dict(
        shuffle=bool(shuffle),
        num_workers=int(num_workers),
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=bool(drop_last),
    )
    if kwargs["num_workers"] > 0:
        kwargs["persistent_workers"] = True
        kwargs["prefetch_factor"] = 2
    return kwargs

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

## Windowing

In [ ]:
def load_and_split(data_path: str):
    """Load merged_dataset.csv and split by time. Return (train, val, test) DataFrames."""
    df = pd.read_csv(data_path, dtype={"location_id": str})
    # Normalize column names (e.g. if CSV uses "temperature_2m_max (°C)" or "DMI EAST")
    col_map = {}
    for c in list(df.columns):
        c_lower = c.strip().lower()
        if "temperature" in c_lower and "max" in c_lower and c != "max_temperature":
            col_map[c] = "max_temperature"
        if "dmi" in c_lower and "east" in c_lower and c != "dmi east":
            col_map[c] = "dmi east"
        if "nino" in c_lower and "3.4" in c_lower and c != "nino anom 3.4":
            col_map[c] = "nino anom 3.4"
    df = df.rename(columns=col_map)
    df["time"] = pd.to_datetime(df["time"])
    df = df.sort_values(["location_id", "time"]).reset_index(drop=True)

    train = df[(df["time"] >= "2005-01-01") & (df["time"] <= "2018-12-31")]
    val   = df[(df["time"] >= "2019-01-01") & (df["time"] <= "2021-12-31")]
    test  = df[(df["time"] >= "2022-01-01") & (df["time"] <= "2025-05-01")]

    return train, val, test

In [ ]:
def build_windows_one_location(times, values, input_len, horizon, stride):
    """Build (X, y) windows for one location. values: (T, n_features)."""
    T = len(times)
    # if the station does not have enough values to form a full window # we skip
    if T < input_len + horizon:
        return None, None
    X_list, y_list = [], []
    for start in range(0, T - input_len - horizon + 1, stride):
        end_in = start + input_len
        end_out = end_in + horizon
        X_list.append(values[start:end_in])
        y_list.append(values[end_in:end_out, 0])
    if not X_list:
        return None, None
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)


def build_windows_temp_transformer_one_location(times, values, input_len, horizon, stride):
    """Build windows for `TemperatureTransformer`.

    Returns:
      - z_past: (input_len, 1) temperature history
      - x_cov_past: (input_len, d_cov) past covariates (features 1:)
      - x_cov_future: (horizon, d_cov) future covariates (features 1:)
      - y: (horizon,) future temperature target (feature 0)

    `values` must be shaped (T, n_features) where feature 0 is temperature.
    """
    T = len(times)
    if T < input_len + horizon:
        return None, None, None, None

    z_past_list, x_cov_past_list, x_cov_future_list, y_list = [], [], [], []

    for start in range(0, T - input_len - horizon + 1, stride):
        end_in = start + input_len
        end_out = end_in + horizon

        past = values[start:end_in]       # (input_len, n_features)
        future = values[end_in:end_out]  # (horizon, n_features)

        z_past_list.append(past[:, :1])
        x_cov_past_list.append(past[:, 1:])
        x_cov_future_list.append(future[:, 1:])
        y_list.append(future[:, 0])

    if not z_past_list:
        return None, None, None, None

    return (
        np.array(z_past_list, dtype=np.float32),
        np.array(x_cov_past_list, dtype=np.float32),
        np.array(x_cov_future_list, dtype=np.float32),
        np.array(y_list, dtype=np.float32),
    )

In [ ]:
def build_all_windows(train, val, test):
    """Build X, y for train/val/test. No cross location_id, no cross split boundary."""
    def run_split(split_df):
        X_list, y_list = [],[]
        grp = grp.sort_values("time")
        mat = grp[FEATURE_COLS].values.astype(np.float32)
        times = grp["time"].values
        X, y = build_windows_one_location(times, mat, INPUT_LEN, HORIZON, STRIDE)
        if not X is not None:
            X_list.append(X)
            y_list.append(y)
        if not X_list:
            return np.zeros((0, INPUT_LEN, len(FEATURE_COLS)), dtype=np.float32), np.zeros((0, INPUT_LEN, len(FEATURE_COLS)), dtype=np.float32)
        return np.concatenate(X_list, axis=0), np.concatenate(y_list, axis=0)
    X_train, y_train = run_split(train)
    X_val,   y_val   = run_split(val)
    X_test,  y_test  = run_split(test)
    return X_train, y_train, X_val, y_val, X_test, y_test

In [ ]:
def build_windows(split_df: pd.DataFrame):
    """Build(X, y) for a single split dataframe (across all location_id)"""
    X_list, y_list = [], []
    for _, grp in split_df.groupby("location_id"):
        grp = grp.sort_values("time")
        mat = grp[FEATURE_COLS].values.astype(np.float32)
        times = grp['time'].values
        X, y = build_windows_one_location(times, mat, INPUT_LEN, HORIZON, STRIDE)
        if X is not None:
            X_list.append(X)
            y_list.append(y)
    if not X_list:
        return (
            np.zeros((0, INPUT_LEN, len(FEATURE_COLS)), dtype=np.fooat32),
            np.zeros((0, HORIZON), dtype=np.float32)
        )
    return np.concatenate(X_list, axis=0), np.concatenate(y_list, axis=0)


def build_windows_temp_transformer(split_df: pd.DataFrame):
    """Build windows for `TemperatureTransformer` across all `location_id`.

    Returns:
      - z_past: (N, INPUT_LEN, 1)
      - x_cov_past: (N, INPUT_LEN, d_cov) where d_cov=len(FEATURE_COLS)-1
      - x_cov_future: (N, HORIZON, d_cov)
      - y: (N, HORIZON) future temperature target
    """
    z_list, x_cov_past_list, x_cov_future_list, y_list = [], [], [], []

    d_cov = len(FEATURE_COLS) - 1

    for _, grp in split_df.groupby("location_id"):
        grp = grp.sort_values("time")
        mat = grp[FEATURE_COLS].values.astype(np.float32)
        times = grp['time'].values

        z_past, x_cov_past, x_cov_future, y = build_windows_temp_transformer_one_location(
            times, mat, INPUT_LEN, HORIZON, STRIDE
        )

        if z_past is not None:
            z_list.append(z_past)
            x_cov_past_list.append(x_cov_past)
            x_cov_future_list.append(x_cov_future)
            y_list.append(y)

    if not z_list:
        return (
            np.zeros((0, INPUT_LEN, 1), dtype=np.float32),
            np.zeros((0, INPUT_LEN, d_cov), dtype=np.float32),
            np.zeros((0, HORIZON, d_cov), dtype=np.float32),
            np.zeros((0, HORIZON), dtype=np.float32),
        )

    return (
        np.concatenate(z_list, axis=0),
        np.concatenate(x_cov_past_list, axis=0),
        np.concatenate(x_cov_future_list, axis=0),
        np.concatenate(y_list, axis=0),
    )

In [ ]:
## Create a boolean fileter to keep only the true rows getting returned
def region_mask(df:pd.DataFrame, region_name: str):
    return df["region"].astype(str).str.strip().str.lower() == region_name.strip().lower()

In [ ]:
train, val, test = load_and_split("../data/merged_dataset.csv")
train_java = train[region_mask(train, "java")]
train_papua = train[region_mask(train, "papua")]
val_papua = val[region_mask(val, "papua")]
test_papua = test[region_mask(test, "papua")]

## Filter Station

In [ ]:
def compute_station_stats(papua_train: pd.DataFrame) -> pd.DataFrame:
    if papua_train is None or len(papua_train) == 0:
        return pd.DataFrame(columns=["location_id", "mean_temp", "std_temp", "elevation", "n_rows"])
    df = papua_train.copy()
    df['location_id'] = df['location_id'].astype(str)
    g = df.groupby("location_id", dropna=False)
    
    stats = g.agg(
        mean_temp=(TARGET_COL, "mean"),
        std_temp=(TARGET_COL, "std"),
        elevation=("elevation", "mean"),
        n_rows=(TARGET_COL, "size"),
    ).reset_index()
    
    stats["mean_temp"] = stats["mean_temp"].astype(np.float32)
    stats["std_temp"] = stats["std_temp"].fillna(0.0).astype(np.float32)
    stats["elevation"] = stats["elevation"].astype(np.float32)
    stats["n_rows"] = stats["n_rows"].astype(int)
    return stats

def select_target_stations_papua_by_elevation(papua_train: pd.DataFrame):
    """
    Select station IDs from Papua training set by elevation:
      - Single station: median elevation station
      - Three stations: lowest, median, highest elevation stations
    Returns (single_station_id: str, three_station_ids: list[str], stats_sorted: pd.DataFrame).
    """
    stats = compute_station_stats(papua_train)
    if len(stats) == 0:
        return None, [], stats

    stats_sorted = stats.sort_values(["elevation", "location_id"], ascending=[True, True]).reset_index(drop=True)
    mid = len(stats_sorted) // 2
    single_id = str(stats_sorted.loc[mid, "location_id"])

    low_id = str(stats_sorted.loc[0, "location_id"])
    high_id = str(stats_sorted.loc[len(stats_sorted) - 1, "location_id"])
    three_ids = [low_id, single_id, high_id]

    # If very small number of stations, deduplicate while keeping order
    seen = set()
    three_ids = [x for x in three_ids if not (x in seen or seen.add(x))]
    return single_id, three_ids, stats_sorted


def filter_df_by_station_ids(df: pd.DataFrame, station_ids) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return df.iloc[0:0].copy() if df is not None else df
    ids = set([str(x) for x in station_ids])
    out = df.copy()
    out["location_id"] = out["location_id"].astype(str)
    return out[out["location_id"].isin(ids)]

In [ ]:
single_id, three_ids, stats_sorted = select_target_stations_papua_by_elevation(train_papua)

In [ ]:
print(f"Papua training stations: {len(stats_sorted)}")
print("Selected station IDs (by elevation):")
print(f"  Single-station (median elevation): {single_id}")
if len(three_ids) == 3:
    print(f"  Three-station (low/median/high):   {three_ids[0]}, {three_ids[1]}, {three_ids[2]}")
else:
    print(f"  Three-station (deduped):           {', '.join(three_ids)}")

## Training Configuration

In [ ]:
def eval_model_mae(model: torch.nn.Module, X: np.ndarray, y: np.ndarray, batch_size: int = 256) -> float:
    if X.shape[0] == 0:
        return float("nan")
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, X.shape[0], batch_size):
            xb = torch.from_numpy(X[i : i + batch_size])
            xb = xb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
            preds.append(model(xb).detach().cpu().numpy().astype(np.float32))
    pred = np.concatenate(preds, axis=0)
    return float(np.mean(np.abs(pred - y)))

In [ ]:
def eval_model_metrics(model: torch.nn.Module, X: np.ndarray, y: np.ndarray, batch_size: int = 256):
    """
    Compute MAE/MSE/RMSE on window targets (y shape: [N, H]).
    Metrics are computed over all elements (N*H).
    Returns dict: {"mae": float, "mse": float, "rmse": float}.
    """
    if X.shape[0] == 0:
        return {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}
    model.eval() # switch model into evaluation mode
    preds = []
    with torch.no_grad(): # disable gradients
        for i in range(0, X.shape[0], batch_size):
            # Extract batch 
            xb = torch.from_numpy(X[i : i + batch_size])
            xb = xb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
            preds.append(model(xb).detach().cpu().numpy().astype(np.float32))
    pred = np.concatenate(preds, axis=0).astype(np.float32)
    y = y.astype(np.float32, copy=False)
    err = (pred - y).astype(np.float32)
    mse = float(np.mean(err ** 2))
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(mse))
    return {"mae": mae, "mse": mse, "rmse": rmse}



## Vanilla Transformer

In [ ]:
import torch.nn as nn

class VanillaTransformer(nn.Module):
    def __init__(self, input_dim=9, d_model=32, nhead=4, num_layers=2, dropout=0.1, horizon=7):
        """
        Model Parameter:
          - input_dim = 9. Number of feature
          - d_model = 32. Internal Feature Size of transformer. 9 Features -> projected to 32 features
          - nhead = 4. Number of Attention Heads
          - num_layers = 2.  Stack 2 Transformer Layers. Each Layer has (self-attention + Feedforward Network)
          - Horizon = 7. Predict 7 days ahead
        """

        super().__init__()
        ## input Projection. Transform from 9 features to 32 Features
        self.input_proj = nn.Linear(input_dim, d_model)
        # transformer decoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
            dropout=dropout, batch_first=True, norm_first=False
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, horizon)

    def forward(self, x):
        # x: (B,T,F)
        x = self.input_proj(x)
        x = self.encoder(x)
        x = x.mean(dim=1)
        return self.head(x)

In [ ]:
def train_supervised_earlystop_target_mae(
    model: nn.Module,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_tgt_val: np.ndarray,
    y_tgt_val: np.ndarray,
    epochs: int = 25,
    batch_size: int = 64,
    lr: float = 1e-3,
    patience: int = 5,
):
    """Train with MSE; early stop on target validation MAE."""
    from torch.utils.data import TensorDataset, DataLoader

    if X_train.shape[0] == 0:
        return model.to(DEVICE), float("nan")

    model = model.to(DEVICE).float()
    train_ds = TensorDataset(torch.from_numpy(X_train).to(DEVICE), torch.from_numpy(y_train).to(DEVICE))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    best_mae, best_state, wait = float("inf"), None, 0

    for ep in range(epochs):
        model.train()
        tr_mse = 0.0
        for xb, yb in train_loader:
            opt.zero_grad() # resetting the gradient from the previous batch. But why?? -> so on the next training iteration the gradient is get set to 0
            pred = model(xb)
            loss = nn.functional.mse_loss(pred, yb)
            loss.backward() # back propagation
            opt.step() # update weights
            tr_mse += loss.item() * xb.size(0) # *** track training loss (accumulates loss accross batches)
        tr_mse /= max(1, X_train.shape[0]) # *** mean training loss per sample

        val_mae = eval_model_mae(model, X_tgt_val, y_tgt_val, batch_size=256)
        print(f"  Epoch {ep+1}/{epochs}  Train MSE: {tr_mse:.6f}  Target Val MAE: {val_mae:.4f}")

        if val_mae < best_mae:
            best_mae = val_mae
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"  Early stopping at epoch {ep+1}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model.to(DEVICE), best_mae


## Domain Adversarial NN

### Feature Extractor

In [ ]:
class FeatureExtractor(nn.Module):
    """Transformer encoder + mean pooling. Output: pooled representation (d_model)."""

    def __init__(self, input_dim: int, d_model: int = 32, nhead: int = 4, num_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=False,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_proj(x)
        x = self.encoder(x)
        return x.mean(dim=1)



### Domain Classifier


In [ ]:
import math
class DomainClassifier(nn.Module):
    """MLP: Linear(in→32) ReLU Linear(32→1) Sigmoid."""

    def __init__(self, in_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(1)  # (B,)


def dann_lambda_schedule(p: float) -> float:
    # λ = 2 / (1 + exp(-10 p)) - 1
    return float(2.0 / (1.0 + math.exp(-10.0 * float(p))) - 1.0)



### Gradient Reversal Layer

In [ ]:
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x: torch.Tensor, lambd: float):
        ctx.lambd = float(lambd)
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        return -ctx.lambd * grad_output, None


def grl(x: torch.Tensor, lambd: float) -> torch.Tensor:
    return GradReverse.apply(x, lambd)


### Training with MAE

In [ ]:
def train_domain_adversarial_dann(
    feat_extractor: nn.Module,
    task_head: nn.Module,
    domain_clf: nn.Module,
    X_src: np.ndarray,
    y_src: np.ndarray,
    X_tgt: np.ndarray,
    y_tgt: np.ndarray,
    X_tgt_val: np.ndarray,
    y_tgt_val: np.ndarray,
    epochs: int = 25,
    batch_size: int = 64,
    lr: float = 1e-3,
    patience: int = 5,
    use_target_task_loss: bool = False,
):
    """
    DANN:
      loss = forecast_MSE + λ * domain_BCE
      λ schedule: 2/(1+exp(-10p)) - 1, p=step/total_steps
      balanced batches: batch_size/2 source + batch_size/2 target
      early stop: target validation MAE
    """
    from torch.utils.data import TensorDataset, DataLoader

    if X_src.shape[0] == 0 or X_tgt.shape[0] == 0:
        return None, float("nan")

    src_bs = max(1, batch_size // 2)
    tgt_bs = max(1, batch_size // 2)

    src_ds = TensorDataset(torch.from_numpy(X_src).to(DEVICE), torch.from_numpy(y_src).to(DEVICE))
    tgt_ds = TensorDataset(torch.from_numpy(X_tgt).to(DEVICE), torch.from_numpy(y_tgt).to(DEVICE))
    src_loader = DataLoader(src_ds, batch_size=src_bs, shuffle=True, num_workers=0, drop_last=True)
    tgt_loader = DataLoader(tgt_ds, batch_size=tgt_bs, shuffle=True, num_workers=0, drop_last=True)

    feat_extractor = feat_extractor.to(DEVICE).float()
    task_head = task_head.to(DEVICE).float()
    domain_clf = domain_clf.to(DEVICE).float()

    params = list(feat_extractor.parameters()) + list(task_head.parameters()) + list(domain_clf.parameters())
    opt = torch.optim.Adam(params, lr=lr)
    bce = nn.BCELoss()

    total_steps = epochs * min(len(src_loader), len(tgt_loader))
    global_step = 0

    best_mae, best_state, wait = float("inf"), None, 0

    for ep in range(epochs):
        feat_extractor.train()
        task_head.train()
        domain_clf.train()

        tr_task, tr_dom = 0.0, 0.0
        n_steps = min(len(src_loader), len(tgt_loader))
        src_iter = iter(src_loader)
        tgt_iter = iter(tgt_loader)

        for _ in range(n_steps):
            xs, ys = next(src_iter)
            xt, yt = next(tgt_iter)
            xb = torch.cat([xs, xt], dim=0)
            dom_y = torch.cat(
                [torch.zeros(xs.size(0), device=DEVICE), torch.ones(xt.size(0), device=DEVICE)],
                dim=0,
            ).float()

            p = global_step / max(1, total_steps)
            lambd = dann_lambda_schedule(p)
            global_step += 1

            opt.zero_grad()
            rep = feat_extractor(xb)
            yhat = task_head(rep)

            task_loss = nn.functional.mse_loss(yhat[: xs.size(0)], ys)
            if use_target_task_loss:
                task_loss = task_loss + nn.functional.mse_loss(yhat[xs.size(0) :], yt)

            dom_pred = domain_clf(grl(rep, lambd))
            dom_loss = bce(dom_pred, dom_y)

            loss = task_loss + (lambd * dom_loss)
            loss.backward()
            opt.step()

            tr_task += float(task_loss.item())
            tr_dom += float(dom_loss.item())

        eval_model = nn.Sequential(feat_extractor, task_head)
        val_mae = eval_model_mae(eval_model, X_tgt_val, y_tgt_val, batch_size=256)
        tr_task /= max(1, n_steps)
        tr_dom /= max(1, n_steps)
        print(f"  Epoch {ep+1}/{epochs}  Task MSE: {tr_task:.6f}  Domain BCE: {tr_dom:.6f}  Target Val MAE: {val_mae:.4f}")

        if val_mae < best_mae:
            best_mae = val_mae
            best_state = {
                "feat": {k: v.cpu().clone() for k, v in feat_extractor.state_dict().items()},
                "task": {k: v.cpu().clone() for k, v in task_head.state_dict().items()},
                "dom": {k: v.cpu().clone() for k, v in domain_clf.state_dict().items()},
            }
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"  Early stopping at epoch {ep+1}")
                break

    if best_state is not None:
        feat_extractor.load_state_dict(best_state["feat"])
        task_head.load_state_dict(best_state["task"])
        domain_clf.load_state_dict(best_state["dom"])

    return nn.Sequential(feat_extractor, task_head).to(DEVICE), best_mae



## KMM

In [ ]:
# KMM domain adaptiation (Java-papua)
def rbf_kernel(X, Y=None, gamma=None):
    if Y is None:
        Y = X
    if gamma is None:
        gamma = 1.0 / (X.shape[1] * X.var() + 1e-8)
    from sklearn.metrics.pairwise import rbf_kernel as sk_rbf
    return sk_rbf(X, Y, gamma=gamma)


def kmm_weights(X_src, X_tgt, B=2.0, eps=0.1, gamma=None):
    """
    Kernel Mean Matching: minimize || (1/n_s) sum_i beta_i phi(x_i) - (1/n_t) sum_j phi(z_j) ||^2
    s.t. 0 <= beta_i <= B, | (1/n_s) sum beta_i - 1 | <= eps.
    """
    n_s, n_t = X_src.shape[0], X_tgt.shape[0]
    K_ss = rbf_kernel(X_src, X_src, gamma=gamma)
    K_st = rbf_kernel(X_src, X_tgt, gamma=gamma)
    kappa = (1.0 / n_t) * K_st.sum(axis=1)
    # min  (1/2) beta' Q beta - kappa' beta   with Q = (1/n_s^2) K_ss
    Q = (1.0 / (n_s * n_s)) * K_ss
    try:
        from cvxopt import matrix, solvers
        solvers.options["show_progress"] = False
        P = matrix(2 * Q)
        q = matrix(-kappa.astype(np.float64))
        G = np.vstack([-np.eye(n_s), np.eye(n_s)])
        h = np.hstack([np.zeros(n_s), B * np.ones(n_s)])
        A = np.ones((1, n_s))
        b = np.array([1.0])
        # | (1/n_s) sum beta - 1 | <= eps  =>  two inequalities
        A2 = np.vstack([A / n_s, -A / n_s])
        b2 = np.array([1 + eps, -(1 - eps)])
        G2 = np.vstack([G, A2])
        h2 = np.hstack([h, b2])
        G2 = matrix(G2.astype(np.float64))
        h2 = matrix(h2.astype(np.float64))
        A = matrix(A.astype(np.float64))
        b = matrix(b.astype(np.float64))
        sol = solvers.qp(P, q, G2, h2, A, b)
        if sol["status"] == "optimal":
            beta = np.array(sol["x"]).ravel().astype(np.float32)
            return np.clip(beta, 0, B)
    except Exception:
        pass
    # Fallback: scipy minimize
    from scipy.optimize import minimize
    def obj(b):
        return 0.5 * b @ Q @ b - kappa @ b

    # | (1/n_s) sum beta - 1 | <= eps  =>  (1-eps) <= sum/n_s <= (1+eps)
    cons = [
        {"type": "ineq", "fun": lambda b: (b.sum() / n_s) - (1 - eps)},
        {"type": "ineq", "fun": lambda b: (1 + eps) - (b.sum() / n_s)},
    ]
    res = minimize(obj, np.ones(n_s), method="SLSQP", bounds=[(0, B)] * n_s, constraints=cons)
    if res.success:
        return np.clip(res.x.astype(np.float32), 0, B)
    return np.ones(n_s, dtype=np.float32)


In [ ]:
DEFAULT_CPU_THREADS = 16
def _maybe_compile(model: torch.nn.Module) -> torch.nn.Module:
    if not USE_TORCH_COMPILE:
        return model
    try:
        return torch.compile(model)  # PyTorch 2.x
    except Exception:
        return model

def _dataloader_kwargs(shuffle: bool, drop_last: bool = False):
    """
    Sensible DataLoader defaults for this workstation.
    - Use CPU workers to overlap preprocessing/host->device transfers.
    - Use pin_memory for faster H2D copies when using CUDA.
    """
    num_workers = 0
    if DEVICE.type == "cuda":
        # Windows multiprocessing can be heavier; keep this conservative but non-zero.
        num_workers = min(8, max(2, 16 // 3))

    kwargs = dict(
        shuffle=bool(shuffle),
        num_workers=int(num_workers),
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=bool(drop_last),
    )
    if kwargs["num_workers"] > 0:
        kwargs["persistent_workers"] = True
        kwargs["prefetch_factor"] = 2
    return kwargs


def train_transformer_weighted(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    weights: np.ndarray,
    epochs: int = 20,
    batch_size: int = 64,
    lr: float = 1e-3,
    patience: int = 5,
):
    """
    Train Transformer with weighted MSE: mean(weights * (pred - target)^2).
    Returns (model, best_val_loss).
    """
    from torch.utils.data import TensorDataset, DataLoader

    # if X_train.shape[0] == 0:
    #     dummy = VanillaTransformer(input_dim=1, d_model=32, nhead=4, num_layers=2, dropout=0.1, horizon=HORIZON)
    #     return _maybe_compile(dummy).to(DEVICE), float("nan")

    w = torch.from_numpy(np.asarray(weights, dtype=np.float32)).float()
    train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train), w[: len(X_train)].clone())
    val_ds = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))
    train_loader = DataLoader(train_ds, batch_size=batch_size, **_dataloader_kwargs(shuffle=True, drop_last=False))
    val_loader = DataLoader(val_ds, batch_size=batch_size, **_dataloader_kwargs(shuffle=False, drop_last=False))
    # train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,num_workers=0)
    # val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=True,num_workers=0)

    model = VanillaTransformer(
        input_dim=X_train.shape[2], d_model=32, nhead=4, num_layers=2,
        dropout=0.1, horizon=HORIZON,
    )
    model = _maybe_compile(model).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best_val, best_state, wait = float("inf"), None, 0
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    for ep in range(epochs):
        model.train()
        train_loss = 0.0
        for xb, yb, wb in train_loader:
            xb = xb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
            yb = yb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
            wb = wb.to(DEVICE, non_blocking=(DEVICE.type == "cuda")).float()
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                pred = model(xb)
                sq = (pred - yb) ** 2
                loss = (wb.unsqueeze(1) * sq).mean()
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            train_loss += loss.item() * xb.size(0)
        train_loss /= len(X_train)

        model.eval()
        val_loss = 0.0
        if len(X_val) > 0:
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb = xb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
                    yb = yb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
                    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                        pred = model(xb)
                        val_loss += nn.functional.mse_loss(pred, yb).item() * xb.size(0)
            val_loss /= len(X_val)
        else:
            val_loss = train_loss

        if (ep + 1) % 5 == 0 or ep == 0:
            print(f"  KMM Epoch {ep+1}/{epochs}  Train loss: {train_loss:.6f}  Val loss: {val_loss:.6f}")
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model.to(DEVICE), float(best_val)



## Climate Aware 

In [ ]:
class ClimateAwareTransformer(nn.Module):
    """
    Temporal local features -> Transformer encoder -> 32-dim pooled
    Geo features (lat, lon, elev) -> Linear(3->16)
    Climate regime (Nino, DMI) -> Linear(2->16)
    Concat -> 64 -> Linear(64->7)
    """

    def __init__(self, horizon: int = 7, d_model: int = 32, nhead: int = 4, num_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.idx_temp = [FEATURE_COLS.index(c) for c in LOCAL_TEMPORAL_COLS]
        self.idx_geo = [FEATURE_COLS.index(c) for c in GEO_COLS]
        self.idx_clim = [FEATURE_COLS.index(c) for c in CLIMATE_COLS]

        self.temporal = FeatureExtractor(input_dim=len(self.idx_temp), d_model=d_model, nhead=nhead, num_layers=num_layers, dropout=dropout)
        self.geo_proj = nn.Linear(3, 16)
        self.clim_proj = nn.Linear(2, 16)
        self.head = nn.Linear(d_model + 16 + 16, horizon)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        xt = x[:, :, self.idx_temp]
        rep_t = self.temporal(xt)
        rep_g = self.geo_proj(x[:, 0, self.idx_geo])
        rep_c = self.clim_proj(x[:, 0, self.idx_clim])
        rep = torch.cat([rep_t, rep_g, rep_c], dim=1)
        return self.head(rep)


class ClimateAwareRep(nn.Module):
    """Same as ClimateAwareTransformer but returns 64-dim representation (for optional DANN)."""

    def __init__(self, d_model: int = 32, nhead: int = 4, num_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.idx_temp = [FEATURE_COLS.index(c) for c in LOCAL_TEMPORAL_COLS]
        self.idx_geo = [FEATURE_COLS.index(c) for c in GEO_COLS]
        self.idx_clim = [FEATURE_COLS.index(c) for c in CLIMATE_COLS]

        self.temporal = FeatureExtractor(input_dim=len(self.idx_temp), d_model=d_model, nhead=nhead, num_layers=num_layers, dropout=dropout)
        self.geo_proj = nn.Linear(3, 16)
        self.clim_proj = nn.Linear(2, 16)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        xt = x[:, :, self.idx_temp]
        rep_t = self.temporal(xt)
        rep_g = self.geo_proj(x[:, 0, self.idx_geo])
        rep_c = self.clim_proj(x[:, 0, self.idx_clim])
        return torch.cat([rep_t, rep_g, rep_c], dim=1)  # (B, 64)

## Climate TFT (Non DA)

In [ ]:
class GRN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim=None, dropout=0.1):
        super().__init__()
        output_dim = output_dim or input_dim

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.elu = nn.ELU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

        self.dropout = nn.Dropout(dropout)
        self.gate = nn.Linear(output_dim, output_dim)
        self.norm = nn.LayerNorm(output_dim)

        if input_dim != output_dim:
            self.skip = nn.Linear(input_dim, output_dim)
        else:
            self.skip = None

    def forward(self, x):
        residual = x if self.skip is None else self.skip(x)

        out = self.fc1(x)
        out = self.elu(out)
        out = self.fc2(out)
        out = self.dropout(out)

        gate = torch.sigmoid(self.gate(out))
        out = gate * out + (1 - gate) * residual

        return self.norm(out)

In [ ]:
class VariableSelection(nn.Module):
    def __init__(self, num_vars, d_model):
        super().__init__()
        self.weight_net = nn.Sequential(
            nn.Linear(num_vars, num_vars),
            nn.Softmax(dim=-1)
        )
        self.proj = nn.Linear(num_vars, d_model)

    def forward(self, x):
        # x: (B, T, num_vars)
        weights = self.weight_net(x.mean(dim=1))  # (B, num_vars)
        x_weighted = x * weights.unsqueeze(1)
        return self.proj(x_weighted)

In [ ]:
class ClimateTFT(nn.Module):
    def __init__(
        self,
        horizon=7,
        d_model=32,
        nhead=4,
        num_layers=2,
        dropout=0.1,
    ):
        super().__init__()

        self.idx_temp = [FEATURE_COLS.index(c) for c in LOCAL_TEMPORAL_COLS]
        self.idx_geo = [FEATURE_COLS.index(c) for c in GEO_COLS]
        self.idx_clim = [FEATURE_COLS.index(c) for c in CLIMATE_COLS]

        # --- Variable Selection ---f
        self.vsn = VariableSelection(len(self.idx_temp), d_model)

        # --- Transformer (same as before but cleaner) ---
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # --- Static encoders (GRN instead of Linear) ---
        self.geo_grn = GRN(3, 32, 16)
        self.clim_grn = GRN(2, 32, 16)

        # --- Temporal GRN ---
        self.temporal_grn = GRN(d_model, 64, d_model)

        # --- Fusion GRN ---
        self.fusion_grn = GRN(d_model + 16 + 16, 64, 64)

        # --- Output ---
        self.head = nn.Linear(64, horizon)

    def forward(self, x):
        # Split inputs
        xt = x[:, :, self.idx_temp]
        xg = x[:, 0, self.idx_geo]
        xc = x[:, 0, self.idx_clim]

        # --- Variable selection ---
        xt = self.vsn(xt)  # (B, T, d_model)

        # --- Transformer ---
        h = self.transformer(xt)

        # Pooling (same as before)
        h = h.mean(dim=1)

        # --- Temporal refinement ---
        h = self.temporal_grn(h)

        # --- Static features ---
        g = self.geo_grn(xg)
        c = self.clim_grn(xc)

        # --- Fusion ---
        rep = torch.cat([h, g, c], dim=1)
        rep = self.fusion_grn(rep)

        return self.head(rep)

## TFT (DA)

### Domain Classifier

In [ ]:
class TFTDomainClassifier(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.net(x)

### Gated Residual Network

In [ ]:
class TFTGRN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim=None, dropout=0.1):
        super().__init__()
        output_dim = output_dim or input_dim

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.elu = nn.ELU()
        self.dropout = nn.Dropout(dropout)

        self.gate = nn.Linear(output_dim, output_dim)
        self.sigmoid = nn.Sigmoid()

        self.skip = nn.Linear(input_dim, output_dim) if input_dim != output_dim else None
        self.layer_norm = nn.LayerNorm(output_dim)

    def forward(self, x):
        residual = x

        x = self.elu(self.fc1(x))
        x = self.dropout(self.fc2(x))

        gate = self.sigmoid(self.gate(x))
        x = gate * x

        if self.skip is not None:
            residual = self.skip(residual)

        return self.layer_norm(x + residual)

### Variable Selection Network

In [ ]:
import torch.nn.functional as F

class TFTVariableSelectionNetwork(nn.Module):
    def __init__(self, input_dim, num_vars, hidden_dim):
        super().__init__()
        self.num_vars = num_vars

        self.var_grns = nn.ModuleList([
            TFTGRN(input_dim, hidden_dim, hidden_dim)
            for _ in range(num_vars)
        ])

        self.weight_grn = TFTGRN(input_dim * num_vars, hidden_dim, num_vars)

    def forward(self, x):
        # x: [B, T, N, D]
        B, T, N, D = x.shape

        var_outputs = []
        for i in range(N):
            var_outputs.append(self.var_grns[i](x[:, :, i, :]))

        var_outputs = torch.stack(var_outputs, dim=2)  # [B, T, N, H]

        flat = x.reshape(B, T, -1)
        weights = F.softmax(self.weight_grn(flat), dim=-1).unsqueeze(-1)

        out = (weights * var_outputs).sum(dim=2)

        return out, weights

### Gradient Reversal Layer

In [ ]:
class TFTGRL(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None

class TFT_DA_Model(nn.Module):
    def __init__(self, tft, hidden_dim):
        super().__init__()
        self.tft = tft

        self.task_head = nn.Linear(hidden_dim, 1)
        self.domain_classifier = TFTDomainClassifier(hidden_dim)

    def forward(self, x, lambda_grl=1.0):
        features, _, _ = self.tft(x)

        # Use last timestep (or pooled)
        feat = features[:, -1, :]

        # Forecast
        y_pred = self.task_head(feat)

        # Domain prediction (with GRL)
        feat_grl = TFTGRL.apply(feat, lambda_grl)
        domain_pred = self.domain_classifier(feat_grl)

        return y_pred, domain_pred

In [ ]:
import torch.nn.functional as F

class TFTVariableSelectionNetwork(nn.Module):
    def __init__(self, input_dim, num_vars, hidden_dim):
        super().__init__()
        self.num_vars = num_vars

        self.var_grns = nn.ModuleList([
            TFTGRN(input_dim, hidden_dim, hidden_dim)
            for _ in range(num_vars)
        ])

        self.weight_grn = TFTGRN(input_dim * num_vars, hidden_dim, num_vars)

    def forward(self, x):
        # x: [B, T, N, D]
        B, T, N, D = x.shape

        var_outputs = []
        for i in range(N):
            var_outputs.append(self.var_grns[i](x[:, :, i, :]))

        var_outputs = torch.stack(var_outputs, dim=2)  # [B, T, N, H]

        flat = x.reshape(B, T, -1)
        weights = F.softmax(self.weight_grn(flat), dim=-1).unsqueeze(-1)

        out = (weights * var_outputs).sum(dim=2)

        return out, weights

### TFT Backbone/Feature Extractor

In [ ]:
class TFT_Backbone(nn.Module):
    def __init__(
        self,
        input_dim,
        num_vars,
        hidden_dim=64,
        lstm_layers=1,
        num_heads=4,
        dropout=0.1
    ):
        super().__init__()

        self.vsn = TFTVariableSelectionNetwork(input_dim, num_vars, hidden_dim)

        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=lstm_layers,
            batch_first=True
        )

        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.post_grn = TFTGRN(hidden_dim, hidden_dim)

    def forward(self, x):
        # x: [B, T, N, D]

        x, var_weights = self.vsn(x)

        lstm_out, _ = self.lstm(x)

        attn_out, attn_weights = self.attn(lstm_out, lstm_out, lstm_out)

        features = self.post_grn(attn_out)

        return features, var_weights, attn_weights

### Full Model

In [ ]:
class TFT_DA_Model(nn.Module):
    def __init__(
        self,
        input_dim,
        num_vars,
        hidden_dim=64,
        horizon=7
    ):
        super().__init__()

        self.horizon = horizon

        self.backbone = TFT_Backbone(
            input_dim=input_dim,
            num_vars=num_vars,
            hidden_dim=hidden_dim
        )

        # Forecast head (multi-horizon)
        self.forecast_head = nn.Linear(hidden_dim, horizon)

        # Domain classifier
        self.domain_classifier = TFTDomainClassifier(hidden_dim)

    def forward(self, x, lambda_grl=1.0):
        features, var_w, attn_w = self.backbone(x)

        # Pooling (important!)
        feat = features.mean(dim=1)  # [B, H]

        # Forecast
        forecast = self.forecast_head(feat)  # [B, horizon]

        # Domain prediction with GRL
        feat_grl = TFTGRL.apply(feat, lambda_grl)
        domain_pred = self.domain_classifier(feat_grl)

        return forecast, domain_pred, var_w, attn_w

In [ ]:
def eval_tft_model_mae(model: torch.nn.Module, X: np.ndarray, y: np.ndarray, batch_size: int = 256) -> float:
    if X.shape[0] == 0:
        return float("nan")

    model.eval()
    preds = []

    with torch.no_grad():
        for i in range(0, X.shape[0], batch_size):
            xb = torch.from_numpy(X[i : i + batch_size])
            xb = xb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))

            # 🔥 FIX HERE
            yhat, _, _, _ = model(xb, lambda_grl=0.0)

            preds.append(yhat.cpu().numpy().astype(np.float32))

    pred = np.concatenate(preds, axis=0)
    return float(np.mean(np.abs(pred - y)))

## Single & Three Station Experiment

In [ ]:
def run_target_station_experiment(
    experiment_name: str,
    train_java_df: pd.DataFrame,
    train_papua_df: pd.DataFrame,
    val_papua_df: pd.DataFrame,
    test_papua_df: pd.DataFrame,
    target_station_ids,
    arima_baseline: dict | None = None,
    epochs: int = 25,
    batch_size: int = 64,
    lr: float = 1e-3,
    patience: int = 5,
):
    """
    Train multiple models for a given target-station subset and evaluate on FULL Papua test set:
      - Vanilla Transformer (Papua-only, supervised)
      - DANN (Java vs selected Papua)
      - KMM (Java reweighted to selected Papua)
      - Climate-Aware Transformer (Papua-only, supervised)
      - Climate-Aware + DANN (Java vs selected Papua)
    Optionally includes a precomputed ARIMA baseline dict.
    """
    print("\n" + "=" * 60)
    print(f"{experiment_name}")
    print("=" * 60)
    print(f"Selected Papua target stations (train only): {', '.join([str(x) for x in target_station_ids])}")

    train_papua_sel = filter_df_by_station_ids(train_papua_df, target_station_ids)

    X_src, y_src = build_windows(train_java_df)
    X_tgt, y_tgt = build_windows(train_papua_sel)
    X_tgt_val, y_tgt_val = build_windows(val_papua_df)  # FULL Papua val
    X_tgt_test, y_tgt_test = build_windows(test_papua_df)  # FULL Papua test

    print(f"Java train windows:         {X_src.shape[0]}")
    print(f"Papua train windows (sel):  {X_tgt.shape[0]}")
    print(f"Papua val windows (FULL):   {X_tgt_val.shape[0]}")
    print(f"Papua test windows (FULL):  {X_tgt_test.shape[0]}")

    if X_tgt.shape[0] == 0 or X_tgt_val.shape[0] == 0 or X_tgt_test.shape[0] == 0:
        print("Not enough windows for this experiment; skipping.")
        out = {
           # "arima": (arima_baseline or {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}),
            "vanilla": {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")},
            "dann": {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")},
            "kmm": {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")},
            #"climate_aware": {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")},
            "climate_aware_dann": {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")},
        }
        return out

    # -----------------------------
    # A) Vanilla: Papua-only (no Java)
    # -----------------------------
    print("\n" + "-" * 60)
    print("A) Vanilla Transformer (Papua only)")
    print("-" * 60)
    vanilla = VanillaTransformer(
        input_dim=X_tgt.shape[2],
        d_model=32,
        nhead=4,
        num_layers=2,
        dropout=0.1,
        horizon=HORIZON,
    )
    vanilla, best_val_mae = train_supervised_earlystop_target_mae(
        vanilla,
        X_tgt,
        y_tgt,
        X_tgt_val,
        y_tgt_val,
        epochs=epochs,
        batch_size=batch_size,
        lr=lr,
        patience=patience,
    )
    vanilla_metrics = eval_model_metrics(vanilla, X_tgt_test, y_tgt_test, batch_size=256)
    print(f"Best Papua Val MAE (early stop): {best_val_mae:.4f}")
    print(
        f"Papua Test  MAE: {vanilla_metrics['mae']:.4f}  "
        f"RMSE: {vanilla_metrics['rmse']:.4f}  MSE: {vanilla_metrics['mse']:.6f}"
    )

    # -----------------------------
    # B) DANN: adversarial DA
    # -----------------------------
    dann_metrics = {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}
    if X_src.shape[0] > 0:
        print("\n" + "-" * 60)
        print("B) DANN (Java vs selected Papua; balanced batches)")
        print("-" * 60)
        feat = FeatureExtractor(input_dim=X_src.shape[2], d_model=32, nhead=4, num_layers=3, dropout=0.1)
        task_head = nn.Linear(32, HORIZON)
        dom = DomainClassifier(in_dim=32)
        dann_model, best_val_mae_dann = train_domain_adversarial_dann(
            feat,
            task_head,
            dom,
            X_src,
            y_src,
            X_tgt,
            y_tgt,
            X_tgt_val,
            y_tgt_val,
            epochs=epochs,
            batch_size=batch_size,
            lr=lr,
            patience=patience,
            use_target_task_loss=False,
        )
        dann_metrics = eval_model_metrics(dann_model, X_tgt_test, y_tgt_test, batch_size=256)
        print(f"Best Papua Val MAE (early stop): {best_val_mae_dann:.4f}")
        print(
            f"Papua Test  MAE: {dann_metrics['mae']:.4f}  "
            f"RMSE: {dann_metrics['rmse']:.4f}  MSE: {dann_metrics['mse']:.6f}"
        )
    else:
        print("\n" + "-" * 60)
        print("B) DANN skipped (no Java/source windows)")
        print("-" * 60)

    # -----------------------------
    # C) KMM: reweight Java -> selected Papua
    # -----------------------------
    kmm_metrics = {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}
    if X_src.shape[0] > 0 and X_tgt.shape[0] > 0:
        print("\n" + "-" * 60)
        print("C) KMM (Java reweighted to selected Papua)")
        print("-" * 60)
        try:
            X_src_flat = X_src.reshape(X_src.shape[0], -1)
            X_tgt_flat = X_tgt.reshape(X_tgt.shape[0], -1)
            subsample_src = min(2000, X_src_flat.shape[0])
            subsample_tgt = min(800, X_tgt_flat.shape[0])
            rs_s = np.random.RandomState(42)
            rs_t = np.random.RandomState(43)
            idx_s = rs_s.choice(X_src_flat.shape[0], subsample_src, replace=False)
            idx_t = rs_t.choice(X_tgt_flat.shape[0], subsample_tgt, replace=False)
            beta = kmm_weights(X_src_flat[idx_s], X_tgt_flat[idx_t], B=2.0, eps=0.1)
            kmm_model, best_val_loss = train_transformer_weighted(
                X_src[idx_s],
                y_src[idx_s],
                X_tgt_val,
                y_tgt_val,
                beta,
                epochs=max(5, min(epochs, 25)),
                batch_size=batch_size,
                lr=lr,
                patience=patience,
            )
            kmm_metrics = eval_model_metrics(kmm_model, X_tgt_test, y_tgt_test, batch_size=256)
            print(f"Best Val loss (early stop): {best_val_loss:.6f}")
            print(
                f"Papua Test  MAE: {kmm_metrics['mae']:.4f}  "
                f"RMSE: {kmm_metrics['rmse']:.4f}  MSE: {kmm_metrics['mse']:.6f}"
            )
        except Exception as e:
            print(f"KMM skipped: {e}")
    else:
        print("\n" + "-" * 60)
        print("C) KMM skipped (no Java/source or no target windows)")
        print("-" * 60)

    # # -----------------------------
    # # D) Climate-Aware: Papua-only supervised
    # # -----------------------------
    # print("\n" + "-" * 60)
    # print("D) Climate-Aware Transformer (Papua only)")
    # print("-" * 60)
    # climate_model = ClimateAwareTransformer(horizon=HORIZON, d_model=32, nhead=4, num_layers=2, dropout=0.1)
    # climate_model, best_val_mae_clim = train_supervised_earlystop_target_mae(
    #     climate_model,
    #     X_tgt,
    #     y_tgt,
    #     X_tgt_val,
    #     y_tgt_val,
    #     epochs=epochs,
    #     batch_size=batch_size,
    #     lr=lr,
    #     patience=patience,
    # )
    # climate_metrics = eval_model_metrics(climate_model, X_tgt_test, y_tgt_test, batch_size=256)
    # print(f"Best Papua Val MAE (early stop): {best_val_mae_clim:.4f}")
    # print(
    #     f"Papua Test  MAE: {climate_metrics['mae']:.4f}  "
    #     f"RMSE: {climate_metrics['rmse']:.4f}  MSE: {climate_metrics['mse']:.6f}"
    # )

    # -----------------------------
    # E) Climate-Aware + DANN
    # -----------------------------
    ca_dann_metrics = {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}
    if X_src.shape[0] > 0:
        print("\n" + "-" * 60)
        print("E) Climate-Aware + DANN (Java vs selected Papua)")
        print("-" * 60)
        ca_feat = ClimateAwareRep(d_model=32, nhead=4, num_layers=2, dropout=0.1)
        ca_task = nn.Linear(64, HORIZON)
        ca_dom = DomainClassifier(in_dim=64)
        ca_dann_model, best_val_mae_cadann = train_domain_adversarial_dann(
            ca_feat,
            ca_task,
            ca_dom,
            X_src,
            y_src,
            X_tgt,
            y_tgt,
            X_tgt_val,
            y_tgt_val,
            epochs=epochs,
            batch_size=batch_size,
            lr=lr,
            patience=patience,
            use_target_task_loss=False,
        )
        ca_dann_metrics = eval_model_metrics(ca_dann_model, X_tgt_test, y_tgt_test, batch_size=256)
        print(f"Best Papua Val MAE (early stop): {best_val_mae_cadann:.4f}")
        print(
            f"Papua Test  MAE: {ca_dann_metrics['mae']:.4f}  "
            f"RMSE: {ca_dann_metrics['rmse']:.4f}  MSE: {ca_dann_metrics['mse']:.6f}"
        )
    else:
        print("\n" + "-" * 60)
        print("E) Climate-Aware + DANN skipped (no Java/source windows)")
        print("-" * 60)

    out = {
        "arima": (arima_baseline or {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}),
        "vanilla": vanilla_metrics,
        "dann": dann_metrics,
        "kmm": kmm_metrics,
        "climate_aware_dann": ca_dann_metrics,
    }

    print("\n" + "-" * 60)
    print("Result (Papua test):")
    for k in ["vanilla", "dann", "kmm", "climate_aware_dann"]:
        m = out[k]
        print(f"  {k:>18s}  MAE={m['mae']:.4f}  MSE={m['mse']:.6f}  RMSE={m['rmse']:.4f}")
    return out



In [ ]:
results_rows = []
def log_row(**kwargs):
    row = {
            "target_region": "papua",
            "source_region": "java",
            "TARGET_COL": TARGET_COL,
            "INPUT_LEN": INPUT_LEN,
            "HORIZON": HORIZON,
            "STRIDE": STRIDE,
            "n_features": len(FEATURE_COLS),
        }
    row.update(kwargs)
    results_rows.append(row)

In [ ]:
res_single = run_target_station_experiment(
        "Single-Station Experiment (Target=Papua; Train target=1 station; Val/Test=FULL Papua)",
        train_java_df=train_java,
        train_papua_df=train_papua,
        val_papua_df=val_papua,
        test_papua_df=test_papua,
        target_station_ids=[single_id],
        arima_baseline=None,
        epochs=25,
        batch_size=64,
        lr=1e-3,
        patienfce=5,
)
log_row(
        experiment="single_station",
        target_station_ids="|".join([str(single_id)]),
        method="vanilla",
        metric_mae=res_single["vanilla"]["mae"],
        metric_mse=res_single["vanilla"]["mse"],
        metric_rmse=res_single["vanilla"]["rmse"],
)
log_row(
        experiment="single_station",
        target_station_ids="|".join([str(single_id)]),
        method="dann",
        metric_mae=res_single["dann"]["mae"],
        metric_mse=res_single["dann"]["mse"],
        metric_rmse=res_single["dann"]["rmse"],
)
log_row(
        experiment="single_station",
        target_station_ids="|".join([str(single_id)]),
        method="kmm",
        metric_mae=res_single["kmm"]["mae"],
        metric_mse=res_single["kmm"]["mse"],
        metric_rmse=res_single["kmm"]["rmse"],
)
log_row(
        experiment="single_station",
        target_station_ids="|".join([str(single_id)]),
        method="climate_aware_dann",
        metric_mae=res_single["climate_aware_dann"]["mae"],
        metric_mse=res_single["climate_aware_dann"]["mse"],
        metric_rmse=res_single["climate_aware_dann"]["rmse"],
)

In [ ]:
beta

## Three Station

In [ ]:
print("PART 4 — THREE-STATION TARGET EXPERIMENT")
print("=" * 60)
res_three = run_target_station_experiment(
        "Three-Station Experiment (Target=Papua; Train target=3 stations; Val/Test=FULL Papua)",
        train_java_df=train_java,
        train_papua_df=train_papua,
        val_papua_df=val_papua,
        test_papua_df=test_papua,
        target_station_ids=three_ids,
        epochs=25,
        batch_size=64,
        lr=1e-3,
        patience=5,
)

log_row(
        experiment="three_station",
        target_station_ids="|".join([str(x) for x in three_ids]),
        method="vanilla",
        metric_mae=res_three["vanilla"]["mae"],
        metric_mse=res_three["vanilla"]["mse"],
        metric_rmse=res_three["vanilla"]["rmse"],
    )
log_row(
        experiment="three_station",
        target_station_ids="|".join([str(x) for x in three_ids]),
        method="dann",
        metric_mae=res_three["dann"]["mae"],
        metric_mse=res_three["dann"]["mse"],
        metric_rmse=res_three["dann"]["rmse"],
)
log_row(
        experiment="three_station",
        target_station_ids="|".join([str(x) for x in three_ids]),
        method="kmm",
        metric_mae=res_three["kmm"]["mae"],
        metric_mse=res_three["kmm"]["mse"],
        metric_rmse=res_three["kmm"]["rmse"],
    )
    
log_row(
        experiment="three_station",
        target_station_ids="|".join([str(x) for x in three_ids]),
        method="climate_aware_dann",
        metric_mae=res_three["climate_aware_dann"]["mae"],
        metric_mse=res_three["climate_aware_dann"]["mse"],
        metric_rmse=res_three["climate_aware_dann"]["rmse"],
)

## TFT DANN TRAINING

In [ ]:
def tft_eval_model_metrics(model, X, y, batch_size=256):
    model.eval()
    preds = []

    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.from_numpy(X[i:i+batch_size]).to(DEVICE)
            yhat, _, _, _ = model(xb, lambda_grl=0.0)
            preds.append(yhat.cpu())

    preds = torch.cat(preds, dim=0)
    y = torch.from_numpy(y)

    mae = torch.mean(torch.abs(preds - y)).item()
    mse = torch.mean((preds - y) ** 2).item()
    rmse = mse ** 0.5

    return {"mae": mae, "mse": mse, "rmse": rmse}

In [ ]:
def train_tft_dann(
    model: nn.Module,
    X_src: np.ndarray,
    y_src: np.ndarray,
    X_tgt: np.ndarray,
    y_tgt: np.ndarray,
    X_tgt_val: np.ndarray,
    y_tgt_val: np.ndarray,
    epochs: int = 100,
    batch_size: int = 256,
    lr: float = 1e-3,
    patience: int = 5,
    use_target_task_loss: bool = False,
):
    from torch.utils.data import TensorDataset, DataLoader

    if X_src.shape[0] == 0 or X_tgt.shape[0] == 0:
        return None, float("nan")

    src_bs = max(1, batch_size // 2)
    tgt_bs = max(1, batch_size // 2)

    src_ds = TensorDataset(torch.from_numpy(X_src).to(DEVICE), torch.from_numpy(y_src).to(DEVICE))
    tgt_ds = TensorDataset(torch.from_numpy(X_tgt).to(DEVICE), torch.from_numpy(y_tgt).to(DEVICE))

    src_loader = DataLoader(src_ds, batch_size=src_bs, shuffle=True, drop_last=True)
    tgt_loader = DataLoader(tgt_ds, batch_size=tgt_bs, shuffle=True, drop_last=True)

    model = model.to(DEVICE).float()
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    total_steps = epochs * min(len(src_loader), len(tgt_loader))
    global_step = 0

    best_mae, best_state, wait = float("inf"), None, 0

    for ep in range(epochs):
        model.train()

        tr_task, tr_dom = 0.0, 0.0
        n_steps = min(len(src_loader), len(tgt_loader))

        src_iter = iter(src_loader)
        tgt_iter = iter(tgt_loader)

        for _ in range(n_steps):
            xs, ys = next(src_iter)
            xt, yt = next(tgt_iter)

            xb = torch.cat([xs, xt], dim=0)

            dom_y = torch.cat([
                torch.zeros(xs.size(0)),
                torch.ones(xt.size(0))
            ]).long().to(DEVICE)

            p = global_step / max(1, total_steps)
            lambd = dann_lambda_schedule(p)
            global_step += 1

            opt.zero_grad()

            # 🔥 TFT forward
            yhat, dom_pred, _, _ = model(xb, lambda_grl=lambd)

            # Task loss
            task_loss = F.mse_loss(yhat[:xs.size(0)], ys)

            if use_target_task_loss:
                task_loss += F.mse_loss(yhat[xs.size(0):], yt)

            # Domain loss
            dom_loss = F.cross_entropy(dom_pred, dom_y)

            loss = task_loss + lambd * dom_loss
            loss.backward()
            opt.step()

            tr_task += task_loss.item()
            tr_dom += dom_loss.item()

        # ✅ Correct evaluation
        val_mae = eval_tft_da(model, X_tgt_val, y_tgt_val)

        tr_task /= max(1, n_steps)
        tr_dom /= max(1, n_steps)

        print(f"Epoch {ep+1}/{epochs} | Task: {tr_task:.6f} | Domain: {tr_dom:.6f} | Val MAE: {val_mae:.4f}")

        if val_mae < best_mae:
            best_mae = val_mae
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        #     wait = 0
        # else:
        #     wait += 1
        #     if wait >= patience:
        #         print(f"Early stopping at epoch {ep+1}")
        #         break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_mae

In [ ]:
def run_tft_dann(
    experiment_name: str,
    train_java_df: pd.DataFrame,
    train_papua_df: pd.DataFrame,
    val_papua_df: pd.DataFrame,
    test_papua_df: pd.DataFrame,
    target_station_ids,
    epochs: int = 25,
    batch_size: int = 64,
    lr: float = 1e-3,
    patience: int = 5,
):
    print("\n" + "=" * 60)
    print(f"{experiment_name}")
    print("=" * 60)
    print(f"Selected Papua target stations (train only): {', '.join([str(x) for x in target_station_ids])}")

    train_papua_sel = filter_df_by_station_ids(train_papua_df, target_station_ids)

    X_src, y_src = build_windows(train_java_df)
    X_tgt, y_tgt = build_windows(train_papua_sel)
    X_tgt_val, y_tgt_val = build_windows(val_papua_df)  # FULL Papua val
    X_tgt_test, y_tgt_test = build_windows(test_papua_df)  # FULL Papua test

    print(f"Java train windows:         {X_src.shape[0]}")
    print(f"Papua train windows (sel):  {X_tgt.shape[0]}")
    print(f"Papua val windows (FULL):   {X_tgt_val.shape[0]}")
    print(f"Papua test windows (FULL):  {X_tgt_test.shape[0]}")

    if X_tgt.shape[0] == 0 or X_tgt_val.shape[0] == 0 or X_tgt_test.shape[0] == 0:
        print("Not enough windows for this experiment; skipping.")
        out = {
           # "arima": (arima_baseline or {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}),
            "vanilla": {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")},
            "dann": {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")},
            "kmm": {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")},
            #"climate_aware": {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")},
            "climate_aware_dann": {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")},
        }
        return out

    # -----------------------------
    # B) TFT - DA
    # -----------------------------
    TFT_dann_metrics = {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}
    if X_src.shape[0] > 0:
        print("\n" + "-" * 60)
        print("B) DANN (Java vs selected Papua; balanced batches)")
        print("-" * 60)
        X_src = X_src.reshape(X_src.shape[0], X_src.shape[1], 9, 1)
        X_tgt = X_tgt.reshape(X_tgt.shape[0], X_tgt.shape[1], 9, 1)
        X_tgt_val = X_tgt_val.reshape(X_tgt_val.shape[0], X_tgt_val.shape[1], 9, 1)
        X_tgt_test = X_tgt_test.reshape(X_tgt_test.shape[0], X_tgt_test.shape[1], 9, 1)
        model = TFT_DA_Model(
            input_dim=X_src.shape[3],
            num_vars=X_src.shape[2],
            hidden_dim=64,
            horizon=HORIZON
        )
    print(X_src.shape)
    TFT_dann_model, best_val_mae = train_tft_dann(
        model,
        X_src,
        y_src,
        X_tgt,
        y_tgt,
        X_tgt_val,
        y_tgt_val,
        epochs=epochs,
        batch_size=batch_size,
        lr=lr,
        patience=patience,
    ) 

    TFT_dann_metrics = tft_eval_model_metrics(TFT_dann_model, X_tgt_test, y_tgt_test, batch_size=256)
    print(f"Best Papua Val MAE (early stop): {best_val_mae:.4f}")
    print(
        f"Papua Test  MAE: {TFT_dann_metrics['mae']:.4f}  "
        f"RMSE: {TFT_dann_metrics['rmse']:.4f}  MSE: {TFT_dann_metrics['mse']:.6f}"
    )
    out = {
        "TFT_dann": TFT_dann_metrics,
    }

    print("\n" + "-" * 60)
    print("Result (Papua test):")
    for k in ["TFT_dann"]:
        m = out[k]
        print(f"  {k:>18s}  MAE={m['mae']:.4f}  MSE={m['mse']:.6f}  RMSE={m['rmse']:.4f}")
    return out



In [ ]:
res_single = run_tft_dann(
        "Single-Station Experiment (Target=Papua; Train target=1 station; Val/Test=FULL Papua)",
        train_java_df=train_java,
        train_papua_df=train_papua,
        val_papua_df=val_papua,
        test_papua_df=test_papua,
        target_station_ids=[single_id],
        epochs=25,
        batch_size=64,
        lr=1e-3,
        patience=5,
)

# WTF IS ALL THIS QUANTILE SHIT

## Train TFT Non-DA

In [ ]:
# train, val, test = load_and_split("../data/merged_dataset.csv")
# train_java = train[region_mask(train, "java")]
# train_papua = train[region_mask(train, "papua")]
# val_papua = val[region_mask(val, "papua")]
# test_papua = test[region_mask(test, "papua")]


X_src, y_src = build_windows(train_java)
X_tgt, y_tgt = build_windows(train_papua)
X_tgt_val, y_tgt_val = build_windows(val_papua)  # FULL Papua val
X_tgt_test, y_tgt_test = build_windows(test_papua)

In [ ]:
X_src.shape

In [ ]:
def run_tft_dann(
    experiment_name: str,
    train_java_df: pd.DataFrame,
    train_papua_df: pd.DataFrame,
    val_papua_df: pd.DataFrame,
    test_papua_df: pd.DataFrame,
    target_station_ids,
    epochs: int = 25,
    batch_size: int = 64,
    lr: float = 1e-3,
    patience: int = 5,
):
    print("\n" + "=" * 60)
    print(f"{experiment_name}")
    print("=" * 60)
    print(f"Selected Papua target stations (train only): {', '.join([str(x) for x in target_station_ids])}")

    train_papua_sel = filter_df_by_station_ids(train_papua_df, target_station_ids)

    X_src, y_src = build_windows(train_java_df)
    X_tgt, y_tgt = build_windows(train_papua_sel)
    X_tgt_val, y_tgt_val = build_windows(val_papua_df)  # FULL Papua val
    X_tgt_test, y_tgt_test = build_windows(test_papua_df)  # FULL Papua test

    print(f"Java train windows:         {X_src.shape[0]}")
    print(f"Papua train windows (sel):  {X_tgt.shape[0]}")
    print(f"Papua val windows (FULL):   {X_tgt_val.shape[0]}")
    print(f"Papua test windows (FULL):  {X_tgt_test.shape[0]}")


    TFT_dann_metrics = {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}
    if X_src.shape[0] > 0:
        print("\n" + "-" * 60)
        print("Temporal Fusion Transformers")
        print("-" * 60)
        TFT = ClimateTFT()
    print(X_src.shape)
    TFT, best_val_mae = train_supervised_earlystop_target_mae(
        TFT,
        X_tgt,
        y_tgt,
        X_tgt_val,
        y_tgt_val,
        epochs=epochs,
        batch_size=batch_size,
        lr=lr,
        patience=patience,
    ) 

    TFT_dann_metrics = eval_model_metrics(TFT, X_tgt_test, y_tgt_test, batch_size=256)
    print(f"Best Papua Val MAE (early stop): {best_val_mae:.4f}")
    print(
        f"Papua Test  MAE: {TFT_dann_metrics['mae']:.4f}  "
        f"RMSE: {TFT_dann_metrics['rmse']:.4f}  MSE: {TFT_dann_metrics['mse']:.6f}"
    )
    out = {
        "TFT_NON_DA": TFT_dann_metrics,
    }

    print("\n" + "-" * 60)
    print("Result (Papua test):")
    for k in ["TFT_NON_DA"]:
        m = out[k]
        print(f"  {k:>18s}  MAE={m['mae']:.4f}  MSE={m['mse']:.6f}  RMSE={m['rmse']:.4f}")
    return out



In [ ]:
res_tft_full = run_tft_dann(
     "Single-Station Experiment: Non DA FULL Papua)",
        train_java_df=train_java,
        train_papua_df=train_papua,
        val_papua_df=val_papua,
        test_papua_df=test_papua,
        target_station_ids=[single_id],
        epochs=50,
        batch_size=64,
        lr=1e-3,
        patience=5
)

## TFT2 NON DA

In [ ]:
class TemporalAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attn = nn.Linear(d_model, 1)

    def forward(self, x):
        # x: (B, T, d_model)
        scores = self.attn(x)                     # (B, T, 1)
        weights = torch.softmax(scores, dim=1)    # attention over time
        context = (weights * x).sum(dim=1)        # weighted sum
        return context, weights

In [ ]:
def quantile_loss(preds, target, quantiles):
    """
    preds: (B, H, Q)
    target: (B, H)
    """
    losses = []
    for i, q in enumerate(quantiles):
        errors = target - preds[:, :, i]
        loss = torch.max((q - 1) * errors, q * errors)
        losses.append(loss.unsqueeze(-1))
    return torch.mean(torch.cat(losses, dim=-1))

In [ ]:
class HorizonDecoder(nn.Module):
    def __init__(self, d_model, horizon, num_quantiles):
        super().__init__()
        self.horizon = horizon
        self.query = nn.Parameter(torch.randn(horizon, d_model))

        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=4,
            batch_first=True
        )

        self.proj = nn.Linear(d_model, num_quantiles)

    def forward(self, context_seq):
        """
        context_seq: (B, T, d_model)
        """
        B = context_seq.size(0)

        # expand learned queries
        q = self.query.unsqueeze(0).expand(B, -1, -1)  # (B, H, d_model)

        # cross-attention: horizon queries attend to time
        out, _ = self.attn(q, context_seq, context_seq)

        return self.proj(out)  # (B, H, Q)

In [ ]:
class ClimateTFTv2(nn.Module):
    def __init__(
        self,
        horizon=7,
        d_model=32,
        nhead=4,
        num_layers=2,
        dropout=0.1,
        quantiles=[0.1, 0.5, 0.9],
    ):
        super().__init__()

        self.quantiles = quantiles
        self.num_q = len(quantiles)

        self.idx_temp = [FEATURE_COLS.index(c) for c in LOCAL_TEMPORAL_COLS]
        self.idx_geo = [FEATURE_COLS.index(c) for c in GEO_COLS]
        self.idx_clim = [FEATURE_COLS.index(c) for c in CLIMATE_COLS]

        # --- Variable Selection ---
        self.vsn = VariableSelection(len(self.idx_temp), d_model)

        # --- Transformer ---
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # --- Attention over time ---
        self.temporal_attn = TemporalAttention(d_model)

        # --- Static encoders ---
        self.geo_grn = GRN(3, 32, d_model)
        self.clim_grn = GRN(2, 32, d_model)

        # --- Fusion ---
        self.fusion_grn = GRN(d_model * 3, 64, d_model)

        # --- Decoder ---
        self.decoder = HorizonDecoder(d_model, horizon, self.num_q)

    def forward(self, x):
        xt = x[:, :, self.idx_temp]
        xg = x[:, 0, self.idx_geo]
        xc = x[:, 0, self.idx_clim]

        # --- Variable selection ---
        xt = self.vsn(xt)

        # --- Transformer ---
        h_seq = self.transformer(xt)

        # --- Temporal attention ---
        h_context, attn_weights = self.temporal_attn(h_seq)

        # --- Static features ---
        g = self.geo_grn(xg)
        c = self.clim_grn(xc)

        # --- Fusion ---
        fused = torch.cat([h_context, g, c], dim=1)
        fused = self.fusion_grn(fused)

        # Expand fused to sequence for decoder
        fused_seq = fused.unsqueeze(1).repeat(1, h_seq.size(1), 1)

        # Combine with temporal sequence
        full_seq = h_seq + fused_seq

        # --- Decoder ---
        out = self.decoder(full_seq)  # (B, H, Q)

        return out, attn_weights

In [ ]:
def tft2_eval_model_metrics(model, X, y, batch_size=256):
    model.eval()
    preds = []

    median_idx = model.quantiles.index(0.5)

    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.from_numpy(X[i:i+batch_size]).to(DEVICE)
            yhat, _ , _= model(xb)

            yhat = yhat[:, :, median_idx]  # (B, H)
            preds.append(yhat.cpu())

    preds = torch.cat(preds, dim=0)
    y = torch.from_numpy(y)

    mae = torch.mean(torch.abs(preds - y)).item()
    mse = torch.mean((preds - y) ** 2).item()
    rmse = mse ** 0.5

    return {"mae": mae, "mse": mse, "rmse": rmse}

In [ ]:
def eval_model_tft2_mae(model, X, y, batch_size=256):
    if X.shape[0] == 0:
        return float("nan")

    model.eval()
    preds = []

    # find median index
    median_idx = model.quantiles.index(0.5)

    with torch.no_grad():
        for i in range(0, X.shape[0], batch_size):
            xb = torch.from_numpy(X[i:i+batch_size]).to(DEVICE)
            y_hat, _ ,_ = model(xb)

            # 👉 take median quantile
            y_hat = y_hat[:, :, median_idx]   # (B, H)

            preds.append(y_hat.cpu().numpy().astype(np.float32))

    pred = np.concatenate(preds, axis=0)  # (N, H)

    return float(np.mean(np.abs(pred - y)))

In [ ]:
def train_tft2_supervised_earlystop_target_mae(
    model: nn.Module,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_tgt_val: np.ndarray,
    y_tgt_val: np.ndarray,
    epochs: int = 25,
    batch_size: int = 64,
    lr: float = 1e-3,
    patience: int = 5,
):
    """Train with MSE; early stop on target validation MAE."""
    from torch.utils.data import TensorDataset, DataLoader

    if X_train.shape[0] == 0:
        return model.to(DEVICE), float("nan")

    model = model.to(DEVICE).float()
    train_ds = TensorDataset(torch.from_numpy(X_train).to(DEVICE), torch.from_numpy(y_train).to(DEVICE))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    best_mae, best_state, wait = float("inf"), None, 0

    for ep in range(epochs):
        model.train()
        tr_mse = 0.0
        for xb, yb in train_loader:
            opt.zero_grad() # resetting the gradient from the previous batch. But why?? -> so on the next training iteration the gradient is get set to 0
            # print(model(xb))
            pred,_  = model(xb)
            loss = quantile_loss(pred, yb, model.quantiles)
            # loss = nn.functional.mse_loss(pred, yb)
            loss.backward() # back propagation
            opt.step() # update weights
            tr_mse += loss.item() * xb.size(0) # *** track training loss (accumulates loss accross batches)
        tr_mse /= max(1, X_train.shape[0]) # *** mean training loss per sample

        val_mae = eval_model_tft2_mae(model, X_tgt_val, y_tgt_val, batch_size=256)
        print(f"  Epoch {ep+1}/{epochs}  Train MSE: {tr_mse:.6f}  Target Val MAE: {val_mae:.4f}")

        if val_mae < best_mae:
            best_mae = val_mae
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"  Early stopping at epoch {ep+1}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model.to(DEVICE), best_mae


In [ ]:
def run_tft2(
    experiment_name: str,
    train_java_df: pd.DataFrame,
    train_papua_df: pd.DataFrame,
    val_papua_df: pd.DataFrame,
    test_papua_df: pd.DataFrame,
    target_station_ids,
    epochs: int = 25,
    batch_size: int = 64,
    lr: float = 1e-3,
    patience: int = 5,
):
    print("\n" + "=" * 60)
    print(f"{experiment_name}")
    print("=" * 60)
    print(f"Selected Papua target stations (train only): {', '.join([str(x) for x in target_station_ids])}")

    train_papua_sel = filter_df_by_station_ids(train_papua_df, target_station_ids)

    X_src, y_src = build_windows(train_java_df)
    X_tgt, y_tgt = build_windows(train_papua_sel)
    X_tgt_val, y_tgt_val = build_windows(val_papua_df)  # FULL Papua val
    X_tgt_test, y_tgt_test = build_windows(test_papua_df)  # FULL Papua test

    print(f"Java train windows:         {X_src.shape[0]}")
    print(f"Papua train windows (sel):  {X_tgt.shape[0]}")
    print(f"Papua val windows (FULL):   {X_tgt_val.shape[0]}")
    print(f"Papua test windows (FULL):  {X_tgt_test.shape[0]}")


    TFT_dann_metrics = {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}
    if X_src.shape[0] > 0:
        print("\n" + "-" * 60)
        print("Temporal Fusion Transformers")
        print("-" * 60)
        TFT2 = ClimateTFTv2()
    print(X_src.shape)
    TFT2, best_val_mae = train_tft2_supervised_earlystop_target_mae(
        TFT2,
        X_tgt,
        y_tgt,
        X_tgt_val,
        y_tgt_val,
        epochs=epochs,
        batch_size=batch_size,
        lr=lr,
        patience=patience,
    ) 

    TFT2_metrics = tft2_eval_model_metrics(TFT2, X_tgt_test, y_tgt_test, batch_size=256)
    print(TFT2_metrics)
    print(f"Best Papua Val MAE (early stop): {best_val_mae:.4f}")
    print(
        f"Papua Test  MAE: {TFT2_metrics['mae']:.4f}  "
        f"RMSE: {TFT2_metrics['rmse']:.4f}  MSE: {TFT2_metrics['mse']:.6f}"
    )
    out = {
        "TFT2_NON_DA": TFT2_metrics,
    }

    print("\n" + "-" * 60)
    print("Result (Papua test):")
    for k in ["TFT2_NON_DA"]:
        m = out[k]
        print(f"  {k:>18s}  MAE={m['mae']:.4f}  MSE={m['mse']:.6f}  RMSE={m['rmse']:.4f}")
    return out



In [ ]:
res_tft2_full = run_tft2(
     "Single-Station Experiment: Non DA FULL Papua)",
        train_java_df=train_java,
        train_papua_df=train_papua,
        val_papua_df=val_papua,
        test_papua_df=test_papua,
        target_station_ids=[single_id],
        epochs=50,
        batch_size=64,
        lr=1e-3,
        patience=5
)

## TFT2 Domain Adapatation

### Gradient Reversal Layer

In [ ]:
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None


def grad_reverse(x, lambda_=1.0):
    return GradReverse.apply(x, lambda_)

### Domain Classifier

In [ ]:
class DomainClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 2)  # source vs target
        )

    def forward(self, x):
        return self.net(x)

### Full Model

In [ ]:
class ClimateTFTv2_DA(ClimateTFTv2):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.domain_clf = DomainClassifier(32)

    def forward(self, x, lambda_grl=0.0):
        xt = x[:, :, self.idx_temp]
        xg = x[:, 0, self.idx_geo]
        xc = x[:, 0, self.idx_clim]

        xt = self.vsn(xt)
        h_seq = self.transformer(xt)

        h_context, attn_weights = self.temporal_attn(h_seq)

        g = self.geo_grn(xg)
        c = self.clim_grn(xc)

        fused = torch.cat([h_context, g, c], dim=1)
        fused = self.fusion_grn(fused)

        # --- Forecast ---
        fused_seq = fused.unsqueeze(1).repeat(1, h_seq.size(1), 1)
        full_seq = h_seq + fused_seq
        forecast = self.decoder(full_seq)

        # --- Domain prediction (GRL applied) ---
        rev_feat = grad_reverse(fused, lambda_grl)
        domain_logits = self.domain_clf(rev_feat)

        return forecast, domain_logits, attn_weights

In [ ]:
def eval_tft2_DA_model_mae(model: torch.nn.Module, X: np.ndarray, y: np.ndarray, batch_size: int = 256) -> float:
    if X.shape[0] == 0:
        return float("nan")

    model.eval()
    preds = []

    with torch.no_grad():
        for i in range(0, X.shape[0], batch_size):
            xb = torch.from_numpy(X[i : i + batch_size])
            xb = xb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
            # 🔥 FIX HERE
            yhat, _ = model(xb, lambda_grl=0.0)

            preds.append(yhat.cpu().numpy().astype(np.float32))

    pred = np.concatenate(preds, axis=0)
    return float(np.mean(np.abs(pred - y)))

In [ ]:
def tft2_eval_model_metrics(model, X, y, batch_size=256):
    model.eval()
    preds = []

    median_idx = model.quantiles.index(0.5)

    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.from_numpy(X[i:i+batch_size]).to(DEVICE)
            yhat, _ , _= model(xb)

            yhat = yhat[:, :, median_idx]  # (B, H)
            preds.append(yhat.cpu())

    preds = torch.cat(preds, dim=0)
    y = torch.from_numpy(y)

    mae = torch.mean(torch.abs(preds - y)).item()
    mse = torch.mean((preds - y) ** 2).item()
    rmse = mse ** 0.5

    return {"mae": mae, "mse": mse, "rmse": rmse}

In [ ]:
def train_dann_tft(
    model,
    X_src, y_src,
    X_tgt,
    X_tgt_val, y_tgt_val,   # 👈 ADD THIS
    epochs=25,
    batch_size=64,
    lambda_max=1.0,
    patience=5
):
    from torch.utils.data import TensorDataset, DataLoader
    import torch.nn.functional as F

    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    src_ds = TensorDataset(torch.from_numpy(X_src), torch.from_numpy(y_src))
    tgt_ds = TensorDataset(torch.from_numpy(X_tgt))

    src_loader = DataLoader(src_ds, batch_size=batch_size,  **_dataloader_kwargs(shuffle=True, drop_last=False))
    tgt_loader = DataLoader(tgt_ds, batch_size=batch_size,  **_dataloader_kwargs(shuffle=True, drop_last=False))

    best_mae = float("inf")
    best_state = None
    wait = 0

    for ep in range(epochs):
        model.train()

        lambda_grl = lambda_max * (ep / epochs)

        for (xs, ys), (xt,) in zip(src_loader, tgt_loader):
            xs, ys = xs.to(DEVICE), ys.to(DEVICE)
            xt = xt.to(DEVICE)

            # --- Forward ---
            pred_s, dom_s, _ = model(xs, lambda_grl)
            _, dom_t, _ = model(xt, lambda_grl)

            # --- Losses ---
            loss_forecast = quantile_loss(pred_s, ys, model.quantiles)

            dom_labels_s = torch.zeros(dom_s.size(0), dtype=torch.long).to(DEVICE)
            dom_labels_t = torch.ones(dom_t.size(0), dtype=torch.long).to(DEVICE)

            loss_domain = (
                F.cross_entropy(dom_s, dom_labels_s) +
                F.cross_entropy(dom_t, dom_labels_t)
            )

            loss = loss_forecast + 0.1 * loss_domain

            # --- Backprop ---
            opt.zero_grad()
            loss.backward()
            opt.step()

        # ✅ VALIDATION (Papua)
        val_mae = eval_model_tft2_mae(model, X_tgt_val, y_tgt_val)

        print(f"Epoch {ep+1} | Val MAE: {val_mae:.4f}")

        # --- Early stopping ---
        if val_mae < best_mae:
            best_mae = val_mae
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping at epoch {ep+1}")
                break

    # restore best model
    if best_state is not None:
        model.load_state_dict(best_state)

    return model.to(DEVICE), best_mae

In [ ]:
def run_tft2_DA(
    experiment_name: str,
    train_java_df: pd.DataFrame,
    train_papua_df: pd.DataFrame,
    val_papua_df: pd.DataFrame,
    test_papua_df: pd.DataFrame
):
    print("\n" + "=" * 60)
    print(f"{experiment_name}")
    

    X_src, y_src = build_windows(train_java_df)
    X_tgt, y_tgt = build_windows(train_papua_df)
    X_tgt_val, y_tgt_val = build_windows(val_papua_df)  # FULL Papua val
    X_tgt_test, y_tgt_test = build_windows(test_papua_df)  # FULL Papua test

    print(f"Java train windows:         {X_src.shape[0]}")
    print(f"Papua train windows (sel):  {X_tgt.shape[0]}")
    print(f"Papua val windows (FULL):   {X_tgt_val.shape[0]}")
    print(f"Papua test windows (FULL):  {X_tgt_test.shape[0]}")


    TFT_dann_metrics = {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}
    if X_src.shape[0] > 0:
        print("\n" + "-" * 60)
        print("Temporal Fusion Transformers V2 Domain Adaptation")
        print("-" * 60)
        TFT2_DA = ClimateTFTv2_DA()
    TFT2_DA, best_val_mae = train_dann_tft(
        TFT2_DA,
        X_src, 
        y_src,
        X_tgt,
        X_tgt_val,
        y_tgt_val,
        epochs=25,
        batch_size=64,
        lambda_max=1.0
    )

    TFT2_DA_metrics = tft2_eval_model_metrics(TFT2_DA, X_tgt_test, y_tgt_test, batch_size=256)
    print(TFT2_DA_metrics)
    print(f"Best Papua Val MAE (early stop): {best_val_mae:.4f}")
    print(
        f"Papua Test  MAE: {TFT2_DA_metrics['mae']:.4f}  "
        f"RMSE: {TFT2_DA_metrics['rmse']:.4f}  MSE: {TFT2_DA_metrics['mse']:.6f}"
    )
    out = {
        "TFT2_DA": TFT2_DA_metrics,
    }

    print("\n" + "-" * 60)
    print("Result (Papua test):")
    for k in ["TFT2_DA"]:
        m = out[k]
        print(f"  {k:>18s}  MAE={m['mae']:.4f}  MSE={m['mse']:.6f}  RMSE={m['rmse']:.4f}")
    return out



In [ ]:
tft2_DA_res = run_tft2_DA(
    experiment_name="Domain Adaptation - TFT New",
    train_java_df=train_java,
    train_papua_df= train_papua,
    val_papua_df= val_papua,
    test_papua_df= test_papua,
)

# KMM Vu Tran

### Projection

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Projection(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x):
        # x: (B, T, in_dim)
        x = self.linear(x)
        x = self.norm(x)
        x = F.gelu(x)
        return x  # (B, T, hidden_dim)

### Full Transformer Model

In [ ]:
class TemperatureTransformer(nn.Module):
    def __init__(
        self,
        input_dim_primary=1,     # temperature
        input_dim_cov=8,         # covariates
        hidden_dim=64,
        n_heads=4,
        num_layers=2,
        forecast_horizon=7
    ):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.forecast_horizon = forecast_horizon

        # 🔹 Projection layers
        self.primary_proj = Projection(input_dim_primary, hidden_dim)
        self.cov_proj = Projection(input_dim_cov, hidden_dim)

        # 🔹 Aggregation projection
        self.agg_proj = Projection(hidden_dim * 2, hidden_dim)

        # 🔹 Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=n_heads,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # 🔹 Transformer Decoder
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=hidden_dim,
            nhead=n_heads,
            batch_first=True
        )
        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=num_layers
        )

        # 🔹 Output layer
        self.output_layer = nn.Linear(hidden_dim, 1)


    def forward(self, z_past, x_cov_past, x_cov_future):
        """
        z_past: (B, T, 1)
        x_cov_past: (B, T, d)
        x_cov_future: (B, tau, d)
        """

        B, T, _ = z_past.shape
        tau = x_cov_future.shape[1]

        # 🔹 1. Projection
        p = self.primary_proj(z_past)            # (B, T, H)
        q_past = self.cov_proj(x_cov_past)       # (B, T, H)
        q_future = self.cov_proj(x_cov_future)   # (B, tau, H)

        # 🔹 2. Concatenate + aggregate
        h = torch.cat([p, q_past], dim=-1)       # (B, T, 2H)
        h = self.agg_proj(h)                     # (B, T, H)

        # 🔹 3. Encoder
        memory = self.encoder(h)                 # (B, T, H)

        # 🔹 4. Decoder (forecast)
        tgt = q_future                           # (B, tau, H)
        out_dec = self.decoder(tgt, memory)      # (B, tau, H)

        forecast = self.output_layer(out_dec)    # (B, tau, 1)

        # 🔹 5. Reconstruction (important!)
        recon_dec = self.decoder(q_past, memory)  # (B, T, H)
        recon = self.output_layer(recon_dec)      # (B, T, 1)

        return forecast, recon

### Compute KMM Old

In [ ]:
def kmm_weights(X_src, X_tgt, B=2.0, eps=0.1, gamma=None):
    """
    Kernel Mean Matching: minimize || (1/n_s) sum_i beta_i phi(x_i) - (1/n_t) sum_j phi(z_j) ||^2
    s.t. 0 <= beta_i <= B, | (1/n_s) sum beta_i - 1 | <= eps.
    """
    n_s, n_t = X_src.shape[0], X_tgt.shape[0]
    K_ss = rbf_kernel(X_src, X_src, gamma=gamma)
    K_st = rbf_kernel(X_src, X_tgt, gamma=gamma)
    kappa = (1.0 / n_t) * K_st.sum(axis=1)
    # min  (1/2) beta' Q beta - kappa' beta   with Q = (1/n_s^2) K_ss
    Q = (1.0 / (n_s * n_s)) * K_ss
    try:
        from cvxopt import matrix, solvers
        solvers.options["show_progress"] = False
        P = matrix(2 * Q)
        q = matrix(-kappa.astype(np.float64))
        G = np.vstack([-np.eye(n_s), np.eye(n_s)])
        h = np.hstack([np.zeros(n_s), B * np.ones(n_s)])
        A = np.ones((1, n_s))
        b = np.array([1.0])
        # | (1/n_s) sum beta - 1 | <= eps  =>  two inequalities
        A2 = np.vstack([A / n_s, -A / n_s])
        b2 = np.array([1 + eps, -(1 - eps)])
        G2 = np.vstack([G, A2])
        h2 = np.hstack([h, b2])
        G2 = matrix(G2.astype(np.float64))
        h2 = matrix(h2.astype(np.float64))
        A = matrix(A.astype(np.float64))
        b = matrix(b.astype(np.float64))
        sol = solvers.qp(P, q, G2, h2, A, b)
        if sol["status"] == "optimal":
            beta = np.array(sol["x"]).ravel().astype(np.float32)
            return np.clip(beta, 0, B)
    except Exception:
        pass
    # Fallback: scipy minimize
    from scipy.optimize import minimize
    def obj(b):
        return 0.5 * b @ Q @ b - kappa @ b

    # | (1/n_s) sum beta - 1 | <= eps  =>  (1-eps) <= sum/n_s <= (1+eps)
    cons = [
        {"type": "ineq", "fun": lambda b: (b.sum() / n_s) - (1 - eps)},
        {"type": "ineq", "fun": lambda b: (1 + eps) - (b.sum() / n_s)},
    ]
    res = minimize(obj, np.ones(n_s), method="SLSQP", bounds=[(0, B)] * n_s, constraints=cons)
    if res.success:
        return np.clip(res.x.astype(np.float32), 0, B)
    return np.ones(n_s, dtype=np.float32)


### Train KMM

In [ ]:
def evaluate_model(model: torch.nn.Module, X: np.ndarray, y: np.ndarray, batch_size: int = 256) -> float:
    if X.shape[0] == 0:
        return float("nan")
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, X.shape[0], batch_size):
            xb = torch.from_numpy(X[i : i + batch_size])
            xb, _ = xb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
            preds.append(model(xb).detach().cpu().numpy().astype(np.float32))
    pred = np.concatenate(preds, axis=0)
    return float(np.mean(np.abs(pred - y)))

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

def train_transformer_weighted_kmm(
    model,
    X_source, y_source,          # Source domain (Java)
    beta,                        # KMM weights (same length as X_source)
    X_val=None, y_val=None,      # Target validation (Papua)
    epochs=20,
    batch_size=256,
    lr=1e-3,
    device="cuda"
):
    """
    Train Transformer with KMM-based domain adaptation.

    X_source: (N, seq_len, features)
    y_source: (N, horizon)
    beta:     (N,) KMM weights
    """

    model.to(device)

    # Convert to tensors
    X_source = torch.from_numpy(X_source).float()
    y_source = torch.from_numpy(y_source).float()
    beta = torch.from_numpy(beta).float()

    # If provided, convert validation arrays once (we will run the same
    # TemperatureTransformer slicing logic during validation).
    if X_val is not None and y_val is not None:
        X_val = torch.from_numpy(X_val).float()
        y_val = torch.from_numpy(y_val).float()

    # Dataset includes weights
    train_ds = TensorDataset(X_source, y_source, beta)
    train_loader = DataLoader(train_ds, batch_size=batch_size,  **_dataloader_kwargs(shuffle=True, drop_last=False))

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Important: we need per-sample loss → no reduction
    criterion = nn.MSELoss(reduction="none")
    best_val_mae, best_state = float('inf'), None
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for xb, yb, wb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            wb = wb.to(device)

            optimizer.zero_grad()

            # TemperatureTransformer expects (z_past, x_cov_past, x_cov_future)
            # Here, xb contains stacked past+future features:
            #   xb[:, :INPUT_LEN, :1]   -> z_past
            #   xb[:, :INPUT_LEN, 1:]   -> x_cov_past
            #   xb[:, INPUT_LEN:, 1:]   -> x_cov_future
            z_past = xb[:, :INPUT_LEN, :1]
            x_cov_past = xb[:, :INPUT_LEN, 1:]
            x_cov_future = xb[:, INPUT_LEN:, 1:]

            forecast, _recon = model(z_past, x_cov_past, x_cov_future)
            preds = forecast.squeeze(-1)  # (B, H)

            # Step 1: per-sample loss
            loss_per_sample = criterion(preds, yb)   # (B, H)

            # Step 2: reduce over horizon → (B,)
            loss_per_sample = loss_per_sample.mean(dim=1)

            # Step 3: apply KMM weights
            weighted_loss = (loss_per_sample * wb).mean()

            weighted_loss.backward()
            optimizer.step()

            total_loss += weighted_loss.item()

        avg_loss = total_loss / len(train_loader)
        # Validation on target domain (IMPORTANT for paper alignment)
        # Validation (unweighted): MAE over all elements (N * H)
        val_abs_sum = 0.0
        val_count = 0
        model.eval()
        with torch.no_grad():
            for i in range(0, X_val.shape[0], 256):
                xb_b = X_val[i : i + 256].to(device)
                yb_b = y_val[i : i + 256].to(device)

                z_past = xb_b[:, :INPUT_LEN, :1]
                x_cov_past = xb_b[:, :INPUT_LEN, 1:]
                x_cov_future = xb_b[:, INPUT_LEN:, 1:]

                forecast, _recon = model(z_past, x_cov_past, x_cov_future)
                pred = forecast.squeeze(-1)  # (B, H)

                val_abs_sum += torch.sum(torch.abs(pred - yb_b)).item()
                val_count += pred.numel()

        val_mae = val_abs_sum / max(1, val_count)
        print(f"Epoch {epoch+1} | Train Loss: {avg_loss:.4f} | Val Loss (Target): {val_mae:.4f}")    
        if best_val_mae < val_mae:
            best_val_mae = val_mae
            best_state = {k : v.cpu().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    return model.to(DEVICE), best_val_mae

In [ ]:
def run_target_station_experiment_kmm(
    experiment_name: str,
    train_java_df: pd.DataFrame,
    train_papua_df: pd.DataFrame,
    val_papua_df: pd.DataFrame,
    test_papua_df: pd.DataFrame,
    beta: np.ndarray,
    target_station_ids,
    epochs: int = 25,
    batch_size: int = 256,
    lr: float = 1e-3, 
    patience: int = 5,
):
    """
    Train multiple models for a given target-station subset and evaluate on FULL Papua test set:
      - KMM (Java reweighted to selected Papua)
    Optionally includes a precomputed ARIMA baseline dict.
    """
    print("\n" + "=" * 60)
    print(f"{experiment_name}")
    print("=" * 60)

    train_selected_sel = filter_df_by_station_ids(train_papua_df, target_station_ids)
    
    X_src, y_src = build_windows(train_java_df)
    X_tgt, y_tgt = build_windows(train_selected_sel)
    X_tgt_val, y_tgt_val = build_windows(val_papua_df)  # FULL Papua val
    X_tgt_test, y_tgt_test = build_windows(test_papua_df)  # FULL Papua test

    

    print(f"Java train windows:         {X_src.shape[0]}")
    print(f"Papua train windows (sel):  {X_tgt.shape[0]}")
    print(f"Papua val windows (FULL):   {X_tgt_val.shape[0]}")
    print(f"Papua test windows (FULL):  {X_tgt_test.shape[0]}")


    # -----------------------------
    # C) KMM: reweight Java -> selected Papua
    # -----------------------------
    kmm_metrics = {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}
    if X_src.shape[0] > 0 and X_tgt.shape[0] > 0:
        print("\n" + "-" * 60)
        print("C) KMM (Java reweighted to selected Papua)")
        print("-" * 60)
        try:
            X_src_flat = X_src.reshape(X_src.shape[0], -1)
            X_tgt_flat = X_tgt.reshape(X_tgt.shape[0], -1)
            subsample_src = min(2000, X_src_flat.shape[0])
            subsample_tgt = min(800, X_tgt_flat.shape[0])
            rs_s = np.random.RandomState(42)
            rs_t = np.random.RandomState(43)
            idx_s = rs_s.choice(X_src_flat.shape[0], subsample_src, replace=False)
            idx_t = rs_t.choice(X_tgt_flat.shape[0], subsample_tgt, replace=False)
            # beta = kmm_weights(X_src_flat[idx_s], X_tgt_flat[idx_t], B=2.0, eps=0.1)
            # Build TemperatureTransformer-compatible windows (past + future covariates).
            # X_*_stack shape: (N, INPUT_LEN + HORIZON, 9) where feature 0 is temperature,
            # and features 1: are the 8 covariates.
            z_src_kmm, x_cov_past_src_kmm, x_cov_future_src_kmm, y_src_kmm = build_windows_temp_transformer(train_java_df)
            z_tgt_val_kmm, x_cov_past_tgt_val_kmm, x_cov_future_tgt_val_kmm, y_tgt_val_kmm = build_windows_temp_transformer(val_papua_df)
            z_tgt_test_kmm, x_cov_past_tgt_test_kmm, x_cov_future_tgt_test_kmm, y_tgt_test_kmm = build_windows_temp_transformer(test_papua_df)

            def _stack_for_temperature(z_past, x_cov_past, x_cov_future):
                zeros_temp_future = np.zeros((z_past.shape[0], HORIZON, 1), dtype=np.float32)
                past = np.concatenate([z_past, x_cov_past], axis=-1)
                future = np.concatenate([zeros_temp_future, x_cov_future], axis=-1)
                return np.concatenate([past, future], axis=1)

            X_src_stack = _stack_for_temperature(z_src_kmm, x_cov_past_src_kmm, x_cov_future_src_kmm)
            X_tgt_val_stack = _stack_for_temperature(z_tgt_val_kmm, x_cov_past_tgt_val_kmm, x_cov_future_tgt_val_kmm)
            X_tgt_test_stack = _stack_for_temperature(z_tgt_test_kmm, x_cov_past_tgt_test_kmm, x_cov_future_tgt_test_kmm)
            model = TemperatureTransformer(input_dim_primary=1,     # temperature
                    input_dim_cov=8,         # covariates
                    hidden_dim=64,
                    n_heads=4,
                    num_layers=2,
                    forecast_horizon=7)
            kmm_model, best_val_loss = train_transformer_weighted_kmm(
                model,
                X_src_stack[idx_s],
                y_src_kmm[idx_s],
                beta,
                X_tgt_val_stack,
                y_tgt_val_kmm,
                epochs=100,
                batch_size=batch_size,
                lr=lr,
                device=DEVICE,
            )

            # --- test metrics for TemperatureTransformer ---
            kmm_model.eval()
            preds = []
            with torch.no_grad():
                for i in range(0, X_tgt_test_stack.shape[0], 256):
                    xb = torch.from_numpy(X_tgt_test_stack[i : i + 256]).float()
                    xb = xb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))

                    z_past = xb[:, :INPUT_LEN, :1]
                    x_cov_past = xb[:, :INPUT_LEN, 1:]
                    x_cov_future = xb[:, INPUT_LEN:, 1:]

                    forecast, _recon = kmm_model(z_past, x_cov_past, x_cov_future)
                    preds.append(forecast.squeeze(-1).detach().cpu().numpy().astype(np.float32))

            pred = np.concatenate(preds, axis=0)
            y_true = y_tgt_test_kmm.astype(np.float32, copy=False)
            err = pred - y_true

            mse = float(np.mean(err ** 2))
            mae = float(np.mean(np.abs(err)))
            rmse = float(np.sqrt(mse))

            kmm_metrics = {"mae": mae, "mse": mse, "rmse": rmse}
            print(f"Best Val loss (early stop): {best_val_loss:.6f}")
            print(
                f"Papua Test  MAE: {kmm_metrics['mae']:.4f}  "
                f"RMSE: {kmm_metrics['rmse']:.4f}  MSE: {kmm_metrics['mse']:.6f}"
            )
        except Exception as e:
            print(f"KMM skipped: {e}")
    else:
        print("\n" + "-" * 60)
        print("C) KMM skipped (no Java/source or no target windows)")
        print("-" * 60)

    
    out = {
       "kmm": kmm_metrics,
    }

    print("\n" + "-" * 60)
    print("Result (Papua test):")
    for k in ["kmm"]:
        m = out[k]
        print(f"  {k:>18s}  MAE={m['mae']:.4f}  MSE={m['mse']:.6f}  RMSE={m['rmse']:.4f}")
    return out


In [ ]:
beta

In [ ]:
res_kmm = run_target_station_experiment_kmm(
        "RUN KMM",
        train_java_df=train_java,
        train_papua_df=train_papua,
        val_papua_df=val_papua,
        test_papua_df=test_papua,
        target_station_ids=[single_id],
        epochs=100,
        batch_size=256,
        lr=1e-3,
        beta=beta,
        patience=5,
)

In [ ]:
res_kmm = run_target_station_experiment_kmm(
        "RUN KMM",
        train_java_df=train_java,
        train_papua_df=train_papua,
        val_papua_df=val_papua,
        test_papua_df=test_papua,
        target_station_ids=three_ids,
        epochs=100,
        batch_size=256,
        lr=1e-3,
        beta=beta,
        patience=5,
)

## Run KMM vanilla transformer

In [ ]:
def run_vanilla_kmm(
    experiment_name: str,
    train_java_df: pd.DataFrame,
    train_papua_df: pd.DataFrame,
    val_papua_df: pd.DataFrame,
    test_papua_df: pd.DataFrame,
    target_station_id,
    epochs: int = 100,
    batch_size: int = 256,
    lr: float = 1e-3,
):
    print("\n" + "=" * 60)
    print(f"{experiment_name}")
    print("=" * 60)
    print(f"Selected Papua target stations (train only): {', '.join([str(x) for x in target_station_ids])}")

    train_papua_sel = filter_df_by_station_ids(train_papua_df, target_station_ids)

    X_src, y_src = build_windows(train_java_df)
    X_tgt, y_tgt = build_windows(train_papua_sel)
    X_tgt_val, y_tgt_val = build_windows(val_papua_df)  # FULL Papua val
    X_tgt_test, y_tgt_test = build_windows(test_papua_df)  # FULL Papua test

    print(f"Java train windows:         {X_src.shape[0]}")
    print(f"Papua train windows (sel):  {X_tgt.shape[0]}")
    print(f"Papua val windows (FULL):   {X_tgt_val.shape[0]}")
    print(f"Papua test windows (FULL):  {X_tgt_test.shape[0]}")

    if X_tgt.shape[0] == 0 or X_tgt_val.shape[0] == 0 or X_tgt_test.shape[0] == 0:
        print("Not enough windows for this experiment; skipping.")
        out = {
           # "arima": (arima_baseline or {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}),
            "vanilla": {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")},
            "dann": {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")},
            "kmm": {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")},
            #"climate_aware": {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")},
            "climate_aware_dann": {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")},
        }
        return out

    # -----------------------------
    # A) Vanilla: Papua-only (no Java)
    # -----------------------------
    print("\n" + "-" * 60)
    print("A) Vanilla Transformer (Papua only)")
    print("-" * 60)
    vanilla = VanillaTransformer(
        input_dim=X_tgt.shape[2],
        d_model=32,
        nhead=4,
        num_layers=2,
        dropout=0.1,
        horizon=HORIZON,
    )
    vanilla, best_val_mae = train_supervised_earlystop_target_mae(
        vanilla,
        X_tgt,
        y_tgt,
        X_tgt_val,
        y_tgt_val,
        epochs=epochs,
        batch_size=batch_size,
        lr=lr,
        patience=patience,
    )
    vanilla_metrics = eval_model_metrics(vanilla, X_tgt_test, y_tgt_test, batch_size=256)
    print(f"Best Papua Val MAE (early stop): {best_val_mae:.4f}")
    print(
        f"Papua Test  MAE: {vanilla_metrics['mae']:.4f}  "
        f"RMSE: {vanilla_metrics['rmse']:.4f}  MSE: {vanilla_metrics['mse']:.6f}"
    )

    # -----------------------------
    # B) DANN: adversarial DA
    # -----------------------------
    dann_metrics = {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}
    if X_src.shape[0] > 0:
        print("\n" + "-" * 60)
        print("B) DANN (Java vs selected Papua; balanced batches)")
        print("-" * 60)
        feat = FeatureExtractor(input_dim=X_src.shape[2], d_model=32, nhead=4, num_layers=3, dropout=0.1)
        task_head = nn.Linear(32, HORIZON)
        dom = DomainClassifier(in_dim=32)
        dann_model, best_val_mae_dann = train_domain_adversarial_dann(
            feat,
            task_head,
            dom,
            X_src,
            y_src,
            X_tgt,
            y_tgt,
            X_tgt_val,
            y_tgt_val,
            epochs=epochs,
            batch_size=batch_size,
            lr=lr,
            patience=patience,
            use_target_task_loss=False,
        )
        dann_metrics = eval_model_metrics(dann_model, X_tgt_test, y_tgt_test, batch_size=256)
        print(f"Best Papua Val MAE (early stop): {best_val_mae_dann:.4f}")
        print(
            f"Papua Test  MAE: {dann_metrics['mae']:.4f}  "
            f"RMSE: {dann_metrics['rmse']:.4f}  MSE: {dann_metrics['mse']:.6f}"
        )
    else:
        print("\n" + "-" * 60)
        print("B) DANN skipped (no Java/source windows)")
        print("-" * 60)

    # -----------------------------
    # C) KMM: reweight Java -> selected Papua
    # -----------------------------
    kmm_metrics = {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}
    if X_src.shape[0] > 0 and X_tgt.shape[0] > 0:
        print("\n" + "-" * 60)
        print("C) KMM (Java reweighted to selected Papua)")
        print("-" * 60)
        try:
            X_src_flat = X_src.reshape(X_src.shape[0], -1)
            X_tgt_flat = X_tgt.reshape(X_tgt.shape[0], -1)
            subsample_src = min(2000, X_src_flat.shape[0])
            subsample_tgt = min(800, X_tgt_flat.shape[0])
            rs_s = np.random.RandomState(42)
            rs_t = np.random.RandomState(43)
            idx_s = rs_s.choice(X_src_flat.shape[0], subsample_src, replace=False)
            idx_t = rs_t.choice(X_tgt_flat.shape[0], subsample_tgt, replace=False)
            beta = kmm_weights(X_src_flat[idx_s], X_tgt_flat[idx_t], B=2.0, eps=0.1)
            kmm_model, best_val_loss = train_transformer_weighted(
                X_src[idx_s],
                y_src[idx_s],
                X_tgt_val,
                y_tgt_val,
                beta,
                epochs=max(5, min(epochs, 25)),
                batch_size=batch_size,
                lr=lr,
                patience=patience,
            )
            kmm_metrics = eval_model_metrics(kmm_model, X_tgt_test, y_tgt_test, batch_size=256)
            print(f"Best Val loss (early stop): {best_val_loss:.6f}")
            print(
                f"Papua Test  MAE: {kmm_metrics['mae']:.4f}  "
                f"RMSE: {kmm_metrics['rmse']:.4f}  MSE: {kmm_metrics['mse']:.6f}"
            )
        except Exception as e:
            print(f"KMM skipped: {e}")
    else:
        print("\n" + "-" * 60)
        print("C) KMM skipped (no Java/source or no target windows)")
        print("-" * 60)

    # # -----------------------------
    # # D) Climate-Aware: Papua-only supervised
    # # -----------------------------
    # print("\n" + "-" * 60)
    # print("D) Climate-Aware Transformer (Papua only)")
    # print("-" * 60)
    # climate_model = ClimateAwareTransformer(horizon=HORIZON, d_model=32, nhead=4, num_layers=2, dropout=0.1)
    # climate_model, best_val_mae_clim = train_supervised_earlystop_target_mae(
    #     climate_model,
    #     X_tgt,
    #     y_tgt,
    #     X_tgt_val,
    #     y_tgt_val,
    #     epochs=epochs,
    #     batch_size=batch_size,
    #     lr=lr,
    #     patience=patience,
    # )
    # climate_metrics = eval_model_metrics(climate_model, X_tgt_test, y_tgt_test, batch_size=256)
    # print(f"Best Papua Val MAE (early stop): {best_val_mae_clim:.4f}")
    # print(
    #     f"Papua Test  MAE: {climate_metrics['mae']:.4f}  "
    #     f"RMSE: {climate_metrics['rmse']:.4f}  MSE: {climate_metrics['mse']:.6f}"
    # )

    # -----------------------------
    # E) Climate-Aware + DANN
    # -----------------------------
    ca_dann_metrics = {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}
    if X_src.shape[0] > 0:
        print("\n" + "-" * 60)
        print("E) Climate-Aware + DANN (Java vs selected Papua)")
        print("-" * 60)
        ca_feat = ClimateAwareRep(d_model=32, nhead=4, num_layers=2, dropout=0.1)
        ca_task = nn.Linear(64, HORIZON)
        ca_dom = DomainClassifier(in_dim=64)
        ca_dann_model, best_val_mae_cadann = train_domain_adversarial_dann(
            ca_feat,
            ca_task,
            ca_dom,
            X_src,
            y_src,
            X_tgt,
            y_tgt,
            X_tgt_val,
            y_tgt_val,
            epochs=epochs,
            batch_size=batch_size,
            lr=lr,
            patience=patience,
            use_target_task_loss=False,
        )
        ca_dann_metrics = eval_model_metrics(ca_dann_model, X_tgt_test, y_tgt_test, batch_size=256)
        print(f"Best Papua Val MAE (early stop): {best_val_mae_cadann:.4f}")
        print(
            f"Papua Test  MAE: {ca_dann_metrics['mae']:.4f}  "
            f"RMSE: {ca_dann_metrics['rmse']:.4f}  MSE: {ca_dann_metrics['mse']:.6f}"
        )
    else:
        print("\n" + "-" * 60)
        print("E) Climate-Aware + DANN skipped (no Java/source windows)")
        print("-" * 60)

    out = {
        "arima": (arima_baseline or {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}),
        "vanilla": vanilla_metrics,
        "dann": dann_metrics,
        "kmm": kmm_metrics,
        "climate_aware_dann": ca_dann_metrics,
    }

    print("\n" + "-" * 60)
    print("Result (Papua test):")
    for k in ["vanilla", "dann", "kmm", "climate_aware_dann"]:
        m = out[k]
        print(f"  {k:>18s}  MAE={m['mae']:.4f}  MSE={m['mse']:.6f}  RMSE={m['rmse']:.4f}")
    return out



## Save Beta Value

In [ ]:
# save beta
np.save("beta_weights.npy", beta)
# load beta
def get_beta(X_source, X_target, gamma):
    filename = ...

    if os.path.exists(filename):
        return np.load(filename)

    beta = compute_kmm(X_source, X_target, gamma)
    np.save(filename, beta)
    return beta